# Credit Risk Modeling — Lending Club Dataset

An end-to-end credit risk pipeline built on Lending Club's accepted-loans dataset (2007–2018Q4, ~2.26M loans). The project follows the standard bank credit-risk stack from the ground up: clean a real, messy lending dataset, model **PD** (Probability of Default), **LGD** (Loss Given Default), and **EAD** (Exposure at Default), combine them into **Expected Loss**, and then layer on the things a bank actually has to do with those numbers — stress testing, **IFRS 9** provisioning, **Basel III** capital, portfolio-level loss simulation, and out-of-time validation.

## What's in here

| Part | Section | Covers |
|---|---|---|
| **I** | Data & Target | Cleaning, leakage removal, missing-value treatment |
| **I** | EDA | Univariate/bivariate analysis, vintage curves, time-to-default |
| **I** | Feature Engineering | WoE encoding, Information Value ranking, derived features |
| **I** | PD Modeling | Logistic regression scorecard + LightGBM (calibrated) |
| **I** | Survival Analysis | Kaplan-Meier curves, Cox Proportional Hazards |
| **I** | Transition Matrix | Markov-chain model of loan delinquency states |
| **I** | LGD Modeling | Two-stage model (logistic "any loss?" + linear "how much?") |
| **I** | EAD & Expected Loss | Amortization-based EAD, portfolio EL by grade/vintage/purpose |
| **II** | Macro & Stress Testing | FRED macro data, time-varying Cox, scenario stress tests |
| **II** | IFRS 9 & Basel III | ECL staging/provisioning, IRB regulatory capital |
| **II** | Portfolio Simulation | Gaussian copula Monte Carlo, VaR / CVaR |
| **II** | Merton Model | Structural credit risk on public-company equity/debt data |
| **III** | Validation | AUC/Gini/KS, calibration, PSI, out-of-time backtest |

**Headline numbers:** LightGBM PD model AUC = 0.713 (Gini 0.425, KS 0.309) vs. a 0.701-AUC logistic scorecard baseline. Two-stage LGD model RMSE = 0.019. Portfolio Expected Loss = **$1.34B on $19.4B** exposure (6.89% EL rate), with Grade G loans running **~22x** the EL rate of Grade A. Out-of-time backtest (train 2013–2016, test 2018) shows AUC holding steady (+0.0013 change) with PSI = 0.025 — no meaningful drift.

## How to read Part II

Parts I and III are a fairly standard, linear modeling pipeline. Part II is written more like a research log on purpose: several of those models (the macro-sensitivity Cox model, the copula correlation parameter, the Distance-to-Default time series) ran into real data limitations along the way, and rather than smoothing that over, this notebook documents **what was tried, why it didn't fully work, and what was used instead** — e.g. a literature-based stress multiplier instead of an unreliable in-sample estimate, or a Basel-prescribed correlation instead of an incomplete calibration. Those notes are left in deliberately, since "here's a limitation and here's how I handled it" is a more honest (and more interesting) artifact than a notebook that pretends everything fit perfectly.

## Setup

```bash
pip install pandas numpy matplotlib seaborn plotly scikit-learn lightgbm lifelines scipy fredapi yfinance requests
```

Dataset: [Lending Club Loan Data](https://www.kaggle.com/datasets/wordsforthewise/lending-club) on Kaggle (`accepted_2007_to_2018Q4.csv.gz`). You'll need a Kaggle API token (`~/.kaggle/kaggle.json`) to download it directly, or download it manually from the link above.


# Part I — Core Credit Risk Pipeline

## 1. Setup & Data Loading

In [ ]:
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)


Download via the Kaggle API (requires a Kaggle account + API token), or skip this cell if you already have the file locally.

In [ ]:
# !pip install kaggle
# !kaggle datasets download -d wordsforthewise/lending-club


In [ ]:
with zipfile.ZipFile("lending-club.zip", "r") as zip_ref:
    zip_ref.extractall("lending_club_data")

df = pd.read_csv("lending_club_data/accepted_2007_to_2018Q4.csv.gz", compression='gzip', low_memory=False)
print(f"Dataset Shape: {df.shape}")

In [ ]:
df.head()

## 2. Target Variable Definition

In [ ]:
df['loan_status'].value_counts(dropna=False)

In [ ]:
# Keep only resolved loans (exclude Current, Late, In Grace Period)
keep = ['Fully Paid', 'Charged Off', 'Default']
df = df[df['loan_status'].isin(keep)]

df['default'] = df['loan_status'].isin(['Charged Off', 'Default']).astype(int)

print(df['default'].value_counts())
print(f"Default rate: {df['default'].mean():.2%}")

## 3. Feature Selection — Drop Leakage & Noise

In [ ]:
# Post-origination performance columns (observed AFTER default — data leakage)
future_leakage_cols = [
    'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv',
    'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee',
    'recoveries', 'collection_recovery_fee', 'last_pymnt_d',
    'last_pymnt_amnt', 'next_pymnt_d',
    'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low',
    'hardship_flag', 'hardship_type', 'hardship_reason', 'hardship_status',
    'deferral_term', 'hardship_amount', 'hardship_start_date', 'hardship_end_date',
    'payment_plan_start_date', 'hardship_length', 'hardship_dpd',
    'hardship_loan_status', 'orig_projected_additional_accrued_interest',
    'hardship_payoff_balance_amount', 'hardship_last_payment_amount',
    'debt_settlement_flag', 'debt_settlement_flag_date', 'settlement_status',
    'settlement_date', 'settlement_amount', 'settlement_percentage', 'settlement_term'
]
administrative_noise = ['id', 'member_id', 'url', 'title', 'zip_code']
columns_to_drop = future_leakage_cols + administrative_noise

df_cleaned = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

print(f"Original attributes: {df.shape[1]}")
print(f"Dropped (leakage + noise): {len([c for c in columns_to_drop if c in df.columns])}")
print(f"Cleaned workspace shape: {df_cleaned.shape}")

## 4. Missing Value Treatment

In [ ]:
missing_stats = df_cleaned.isnull().mean() * 100
missing_stats = missing_stats[missing_stats > 0].sort_values(ascending=False)
print("Features with Missing Data (%):")
print(missing_stats.head(25))

### Missing Value Treatment Decisions

| Feature Group | Missing % | Interpretation | Treatment |
|---|---|---|---|
| Joint application (`sec_app_*`) | ~98% | Non-joint loans: no co-borrower | Create `is_joint_application` flag; drop `sec_app_*` |
| Derogatory history (`mths_since_*`) | 66–83% | Missing = event never occurred | Fill 999 sentinel; create `ever_*` binary flag |
| Installment utilization (`il_util`) | 60–65% | Missing = no installment accounts | Create `has_installment_account`; fill 0 |
| Loan description (`desc`) | 91% | Optional free text | Create `has_desc` presence flag |

**Key insight:** In credit modeling, *missing = never* is often the correct interpretation for derogatory fields. A borrower with `mths_since_last_major_derog = NaN` has no derogatory history — that is good credit behavior, not missing data.

In [ ]:
# GROUP 1: Joint application
df_cleaned['is_joint_application'] = df_cleaned['annual_inc_joint'].notna().astype(int)
joint_cols = [c for c in df_cleaned.columns if c.startswith('sec_app_')]
joint_cols += ['verification_status_joint', 'dti_joint', 'annual_inc_joint', 'revol_bal_joint']
df_cleaned.drop(columns=[col for col in joint_cols if col in df_cleaned.columns], inplace=True)

# GROUP 2: "Never happened" derogatory fields
never_event_cols = [
    'mths_since_last_record', 'mths_since_recent_bc_dlq',
    'mths_since_last_major_derog', 'mths_since_recent_revol_delinq'
]
for col in never_event_cols:
    if col in df_cleaned.columns:
        df_cleaned[f'ever_{col.replace("mths_since_", "")}'] = df_cleaned[col].notna().astype(int)
        df_cleaned[col] = df_cleaned[col].fillna(999)

# GROUP 3: Installment/utilization
if 'il_util' in df_cleaned.columns:
    df_cleaned['has_installment_account'] = df_cleaned['il_util'].notna().astype(int)
    df_cleaned['il_util'] = df_cleaned['il_util'].fillna(0)
if 'all_util' in df_cleaned.columns:
    df_cleaned['all_util'] = df_cleaned['all_util'].fillna(0)
if 'open_acc_6m' in df_cleaned.columns:
    df_cleaned['open_acc_6m'] = df_cleaned['open_acc_6m'].fillna(0)
if 'mths_since_rcnt_il' in df_cleaned.columns:
    df_cleaned['ever_had_installment'] = df_cleaned['mths_since_rcnt_il'].notna().astype(int)
    df_cleaned['mths_since_rcnt_il'] = df_cleaned['mths_since_rcnt_il'].fillna(999)

# GROUP 4: desc
if 'desc' in df_cleaned.columns:
    df_cleaned['has_desc'] = df_cleaned['desc'].notna().astype(int)

print("Missing value transformations complete.")
print(f"Current workspace shape: {df_cleaned.shape}")

## 5. Exploratory Data Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Default rate by categorical features
features_to_plot = ['purpose', 'emp_length', 'home_ownership', 'addr_state']
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.flatten()

for i, col in enumerate(features_to_plot):
    if col in df_cleaned.columns:
        risk_rates = df_cleaned.groupby(col)['default'].mean().sort_values(ascending=False).reset_index()
        risk_rates['default'] = risk_rates['default'] * 100
        sns.barplot(data=risk_rates, x=col, y='default', ax=axes[i], palette='mako', hue=col, legend=False)
        axes[i].set_title(f'Default Rate by {col.replace("_", " ").title()}', fontsize=14)
        axes[i].set_ylabel('Default Rate (%)')
        axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
df_cleaned.groupby('grade')['default'].mean().sort_values().plot(kind='bar',
    title='Default Rate by Grade', ylabel='Default Rate')
plt.tight_layout()
plt.show()

In [ ]:
import plotly.express as px
state_dr = df_cleaned.groupby('addr_state')['default'].mean().reset_index()
state_dr['default_rate_pct'] = state_dr['default'] * 100
fig = px.choropleth(state_dr, locations='addr_state', locationmode='USA-states',
                    color='default_rate_pct', scope='usa',
                    labels={'default_rate_pct': 'Default Rate (%)'},
                    title='Default Rate by State')
fig.show()

### Numerical Feature Distribution Analysis

Visualize how key continuous features separate defaulters (red) from non-defaulters (blue).

In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()
palette_colors = {0: "#1f77b4", 1: "#d62728"}

if 'fico_range_low' in df_cleaned.columns:
    sns.kdeplot(data=df_cleaned, x='fico_range_low', hue='default',
                common_norm=False, fill=True, alpha=0.4, palette=palette_colors, ax=axes[0])
    axes[0].set_title('FICO Score Distribution', fontsize=13, fontweight='bold')

if 'int_rate' in df_cleaned.columns:
    sns.kdeplot(data=df_cleaned, x='int_rate', hue='default',
                common_norm=False, fill=True, alpha=0.4, palette=palette_colors, ax=axes[1])
    axes[1].set_title('Interest Rate Distribution', fontsize=13, fontweight='bold')

if 'dti' in df_cleaned.columns:
    sns.kdeplot(data=df_cleaned[df_cleaned['dti'] <= 50], x='dti', hue='default',
                common_norm=False, fill=True, alpha=0.4, palette=palette_colors, ax=axes[2])
    axes[2].set_title('DTI Ratio (Capped at 50)', fontsize=13, fontweight='bold')

if 'annual_inc' in df_cleaned.columns:
    sns.kdeplot(data=df_cleaned[df_cleaned['annual_inc'] > 0], x='annual_inc', hue='default',
                common_norm=False, fill=True, alpha=0.4, palette=palette_colors, log_scale=True, ax=axes[3])
    axes[3].set_title('Annual Income (Log Scale)', fontsize=13, fontweight='bold')

if 'loan_amnt' in df_cleaned.columns:
    sns.histplot(data=df_cleaned, x='loan_amnt', hue='default',
                 multiple="layer", bins=35, palette=palette_colors, ax=axes[4], element="step", fill=True, alpha=0.3)
    axes[4].set_title('Loan Amount Distribution', fontsize=13, fontweight='bold')

fig.delaxes(axes[5])
plt.tight_layout()
plt.show()

**Key observations:**
- **Interest Rate** has the best separation. Defaulters skew right (15–25%), paid loans skew left (5–12%).
- **FICO Score:** Default peaks ~650–670, paid peaks ~700+. Strong signal.
- **DTI:** Mild separation — defaulters shift slightly right.
- **Annual Income:** Partial separation; log scale needed due to skew.

In [ ]:
core_numeric_cols = [
    'loan_amnt', 'int_rate', 'installment', 'annual_inc', 'dti',
    'fico_range_low', 'fico_range_high', 'open_acc', 'pub_rec',
    'revol_bal', 'revol_util', 'total_acc', 'default'
]
available_cols = [col for col in core_numeric_cols if col in df_cleaned.columns]
correlation_matrix = df_cleaned[available_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap='coolwarm',
            vmin=-1, vmax=1, center=0, square=True, linewidths=0.5, cbar_kws={"shrink": .8})
plt.title('Correlation Heatmap: Continuous Risk Features', fontsize=14, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

**Key insights:** `fico_range_low` and `fico_range_high` are perfectly collinear (r=1.0) — drop one. `loan_amnt` and `installment` are highly correlated (~0.95) — drop `installment`.

## 6. Macro Credit Risk — Vintage Analysis

Vintage analysis tracks cohorts of loans by origination period and compares their default trajectories.

**Why it matters:** If a new vintage's cumulative default curve is rising faster than previous vintages at the same loan age — that's an early warning signal. IFRS 9 uses this logic for lifetime PD estimation.

In [ ]:
df_cleaned['issue_d'] = pd.to_datetime(df_cleaned['issue_d'])
df_cleaned['issue_year'] = df_cleaned['issue_d'].dt.year
df_cleaned['issue_quarter'] = df_cleaned['issue_d'].dt.to_period('Q')

fig, axes = plt.subplots(2, 1, figsize=(15, 14))

# Part 1: Quarterly macro default rate
quarterly_data = df_cleaned.groupby('issue_quarter')['default'].mean() * 100
sns.lineplot(x=quarterly_data.index.to_timestamp(), y=quarterly_data.values,
             ax=axes[0], color='#2c3e50', linewidth=2.5, marker='o')
axes[0].set_title('Historical Macro Default Rate by Origination Quarter', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Default Rate (%)')
axes[0].grid(True, linestyle='--', alpha=0.6)

# Part 2: Cumulative lifecycle curves per vintage
if 'last_pymnt_d' in df_cleaned.columns:
    df_cleaned['last_pymnt_d'] = pd.to_datetime(df_cleaned['last_pymnt_d'])
    df_cleaned['months_since_issue'] = (
        (df_cleaned['last_pymnt_d'].dt.year - df_cleaned['issue_d'].dt.year) * 12 +
        (df_cleaned['last_pymnt_d'].dt.month - df_cleaned['issue_d'].dt.month)
    ).clip(0, 36)
else:
    df_cleaned['months_since_issue'] = df_cleaned['grade'].map(
        {'A':5, 'B':10, 'C':15, 'D':20, 'E':25, 'F':30, 'G':36})

vintage_years = sorted([y for y in df_cleaned['issue_year'].unique() if 2011 <= y <= 2018])
colors = sns.color_palette("viridis", len(vintage_years))
months_axis = sorted(df_cleaned['months_since_issue'].dropna().unique())

for idx, year in enumerate(vintage_years):
    cohort = df_cleaned[df_cleaned['issue_year'] == year]
    total = len(cohort)
    if total > 5000:
        running = 0
        rates = []
        for m in months_axis:
            running += len(cohort[(cohort['months_since_issue'] == m) & (cohort['default'] == 1)])
            rates.append(running / total * 100)
        axes[1].plot(months_axis, rates, marker='x', label=f'{year}', color=colors[idx], linewidth=2)

axes[1].set_title('Cumulative Default Curves by Origination Vintage', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Months Elapsed Since Origination')
axes[1].set_ylabel('Cumulative Default Rate (%)')
axes[1].legend(title='Vintage Year', bbox_to_anchor=(1.02, 1), loc='upper left')
axes[1].grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

## 7. Credit Asset Duration — Time-to-Default Distribution

In [ ]:
# Use original df (last_pymnt_d was dropped from df_cleaned to prevent leakage)
defaulted_loans = df[df['default'] == 1].copy()
defaulted_loans['issue_d'] = pd.to_datetime(defaulted_loans['issue_d'])
defaulted_loans['last_pymnt_d'] = pd.to_datetime(defaulted_loans['last_pymnt_d'])

defaulted_loans['months_to_default'] = (
    (defaulted_loans['last_pymnt_d'].dt.year - defaulted_loans['issue_d'].dt.year) * 12 +
    (defaulted_loans['last_pymnt_d'].dt.month - defaulted_loans['issue_d'].dt.month)
)
defaulted_loans = defaulted_loans[
    (defaulted_loans['months_to_default'] >= 0) &
    (defaulted_loans['months_to_default'] <= 60)
]

median_time = defaulted_loans['months_to_default'].median()
mean_time = defaulted_loans['months_to_default'].mean()

plt.figure(figsize=(12, 6))
sns.histplot(data=defaulted_loans, x='months_to_default', bins=45, kde=True,
             color='#d62728', alpha=0.6, edgecolor='white')
plt.axvline(median_time, color='black', linestyle='--', linewidth=2, label=f'Median: {median_time:.1f}m')
plt.axvline(mean_time, color='blue', linestyle=':', linewidth=2, label=f'Mean: {mean_time:.1f}m')
plt.title('Time-to-Default Distribution (Charged-Off Population)', fontsize=14, fontweight='bold')
plt.xlabel('Months Elapsed from Origination to Final Payment')
plt.ylabel('Count')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print(f"Median Time-to-Default: {median_time:.1f} months")
print(f"Mean Time-to-Default:   {mean_time:.1f} months")

## 8. Feature Engineering — WOE Encoding & IV Ranking

In [ ]:
def calculate_woe_iv(dataframe, feature, target_col='default'):
    """Computes Weight of Evidence (WoE) and Information Value (IV) for a categorical column."""
    total_goods = (dataframe[target_col] == 0).sum()
    total_bads  = (dataframe[target_col] == 1).sum()

    df_counts = dataframe.groupby(feature)[target_col].agg(total_count='count', bads='sum')
    df_counts['goods'] = df_counts['total_count'] - df_counts['bads']
    df_counts['pct_goods'] = (df_counts['goods'] + 0.5) / total_goods  # epsilon avoids log(0)
    df_counts['pct_bads']  = (df_counts['bads']  + 0.5) / total_bads
    df_counts[f'{feature}_WoE'] = np.log(df_counts['pct_goods'] / df_counts['pct_bads'])
    df_counts['IV_component'] = (df_counts['pct_goods'] - df_counts['pct_bads']) * df_counts[f'{feature}_WoE']
    total_iv = df_counts['IV_component'].sum()

    print(f"--- {feature.upper()} (IV: {total_iv:.4f}) ---")
    print(df_counts[[f'{feature}_WoE', 'total_count']].round(4))
    print("-" * 50)

    return df_counts[f'{feature}_WoE'].to_dict(), total_iv


categorical_features = ['grade', 'purpose', 'home_ownership', 'emp_length']
iv_tracker = {}

for feature in categorical_features:
    if feature in df_cleaned.columns:
        woe_map, iv = calculate_woe_iv(df_cleaned, feature)
        iv_tracker[feature] = iv
        df_cleaned[f'{feature}_woe'] = df_cleaned[feature].map(woe_map)

print("\nCategorical WoE encoding complete.")

In [ ]:
def calculate_numerical_iv(dataframe, feature, target_col='default', bins=10):
    """Bins a numerical feature and computes its IV."""
    try:
        dataframe['binned'] = pd.qcut(dataframe[feature], q=bins, duplicates='drop')
        _, iv = calculate_woe_iv(dataframe, 'binned', target_col)
        dataframe.drop(columns=['binned'], inplace=True)
        return iv
    except Exception as e:
        print(f"Skipping {feature}: {e}")
        return None


numerical_features = [
    'loan_amnt', 'int_rate', 'installment', 'annual_inc', 'dti',
    'fico_range_low', 'fico_range_high', 'open_acc', 'pub_rec',
    'revol_bal', 'revol_util', 'total_acc', 'mort_acc'
]
for feature in numerical_features:
    if feature in df_cleaned.columns:
        iv = calculate_numerical_iv(df_cleaned, feature)
        if iv is not None:
            iv_tracker[feature] = iv

iv_summary = pd.DataFrame.from_dict(iv_tracker, orient='index', columns=['IV'])
iv_summary = iv_summary.sort_values('IV', ascending=False)
iv_summary['Predictive_Power'] = pd.cut(
    iv_summary['IV'],
    bins=[-np.inf, 0.02, 0.1, 0.3, 0.5, np.inf],
    labels=['Useless', 'Weak', 'Medium', 'Strong', 'Very Strong']
)
print(iv_summary)
print(f"\nFeatures to DROP (IV < 0.02): {iv_summary[iv_summary['IV'] < 0.02].index.tolist()}")

**IV Results:** `grade` is the single strongest categorical predictor (IV=0.46). `emp_length` is dropped (IV=0.002). `purpose` is borderline — keep WoE encoding but expect low model contribution.

**Final feature set:** `grade_woe`, `int_rate`, `fico_range_low`, `dti`, `mort_acc`, `loan_amnt`, `home_ownership_woe`, `annual_inc`, `revol_util`, `has_pub_rec`, `loan_to_income`

In [ ]:
print(f"Starting shape: {df_cleaned.shape}")

# Create binary flag before dropping raw pub_rec
if 'pub_rec' in df_cleaned.columns:
    df_cleaned['has_pub_rec'] = (df_cleaned['pub_rec'] > 0).astype(int)

explicit_drop_list = [
    'purpose', 'purpose_woe', 'open_acc', 'revol_bal',
    'emp_length', 'emp_length_woe', 'total_acc',
    'pub_rec',           # replaced by binary flag
    'fico_range_high',   # redundant with fico_range_low (r=1.0)
    'installment'        # redundant with loan_amnt (r=0.95)
]
columns_to_remove = [col for col in explicit_drop_list if col in df_cleaned.columns]
df_cleaned.drop(columns=columns_to_remove, inplace=True, errors='ignore')

print(f"Dropped: {columns_to_remove}")
print(f"Final shape after pruning: {df_cleaned.shape}")

In [ ]:
# 1. LOAN-TO-INCOME RATIO
df_cleaned['loan_to_income'] = df_cleaned['loan_amnt'] / df_cleaned['annual_inc'].clip(lower=1)

# 2. CREDIT UTILIZATION (fallback to revol_util if revol_bal was dropped)
if 'revol_bal' in df_cleaned.columns and 'total_rev_hi_lim' in df_cleaned.columns:
    df_cleaned['credit_util_calculated'] = (df_cleaned['revol_bal'] /
                                             df_cleaned['total_rev_hi_lim'].clip(lower=1))
else:
    print("→ Using baseline revol_util (revol_bal was dropped earlier).")

# 3. DEROGATORY COMPOSITE SCORE
delinq = df_cleaned['delinq_2yrs'].fillna(0)
pub_rec_flag = df_cleaned['has_pub_rec'].fillna(0)
pub_rec_bankruptcies = df_cleaned['pub_rec_bankruptcies'].fillna(0) if 'pub_rec_bankruptcies' in df_cleaned.columns else 0
df_cleaned['derog_composite'] = delinq + pub_rec_flag + pub_rec_bankruptcies

# 4. CREDIT AGE IN MONTHS
if 'earliest_cr_line' in df_cleaned.columns:
    df_cleaned['earliest_cr_line'] = pd.to_datetime(df_cleaned['earliest_cr_line'], format='mixed')
    df_cleaned['issue_d'] = pd.to_datetime(df_cleaned['issue_d'], format='mixed')
    df_cleaned['credit_age_months'] = (
        (df_cleaned['issue_d'].dt.year - df_cleaned['earliest_cr_line'].dt.year) * 12 +
        (df_cleaned['issue_d'].dt.month - df_cleaned['earliest_cr_line'].dt.month)
    ).clip(lower=0)

print(f"Shape after feature engineering: {df_cleaned.shape}")

In [ ]:
for feat in ['loan_to_income', 'derog_composite', 'credit_age_months']:
    if feat in df_cleaned.columns:
        iv = calculate_numerical_iv(df_cleaned, feat)
        print(f"{feat}: IV = {iv:.4f}")

In [ ]:
# Drop derived features with low IV
df_cleaned.drop(columns=['derog_composite', 'credit_age_months'], inplace=True, errors='ignore')
print(f"Final modeling shape: {df_cleaned.shape}")

**Future work — LLM text feature extraction:** the `desc` column (free-text loan description, ~9% non-null) could be run through an LLM on a sample of loans to extract structured features such as a financial-stress score, purpose clarity, tone, or red flags — deferred here rather than implemented.

## 9. PD Modeling — Logistic Regression Scorecard

In [ ]:
from sklearn.model_selection import train_test_split

feature_cols = [
    'grade_woe', 'home_ownership_woe',
    'int_rate', 'fico_range_low', 'dti', 'mort_acc',
    'loan_amnt', 'annual_inc', 'revol_util', 'loan_to_income', 'has_pub_rec',
]
target_col = 'default'

model_df = df_cleaned[feature_cols + [target_col]].dropna()
print(f"Modeling dataset shape: {model_df.shape}")
print(f"Default rate: {model_df[target_col].mean():.4f}")

X = model_df[feature_cols]
y = model_df[target_col]

# 70 / 15 / 15 stratified split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f"\nTrain:      {X_train.shape[0]:>8,} rows | Default rate: {y_train.mean():.4f}")
print(f"Validation: {X_val.shape[0]:>8,} rows | Default rate: {y_val.mean():.4f}")
print(f"Test:       {X_test.shape[0]:>8,} rows | Default rate: {y_test.mean():.4f}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import roc_auc_score, brier_score_loss

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

lr_model = LogisticRegression(max_iter=1000, random_state=42, solver='lbfgs', class_weight='balanced')
lr_model.fit(X_train_scaled, y_train)

print("Coefficients after scaling:")
print(pd.Series(lr_model.coef_[0], index=feature_cols).sort_values(ascending=False))

raw_probs_val = lr_model.predict_proba(X_val_scaled)[:, 1]
print(f"\nRaw AUC on Validation: {roc_auc_score(y_val, raw_probs_val):.4f}")

# Platt scaling
calibrated_model = CalibratedClassifierCV(lr_model, method='sigmoid', cv='prefit')
calibrated_model.fit(X_val_scaled, y_val)

cal_probs_val  = calibrated_model.predict_proba(X_val_scaled)[:, 1]
cal_probs_test = calibrated_model.predict_proba(X_test_scaled)[:, 1]

print(f"AUC Validation: {roc_auc_score(y_val, cal_probs_val):.4f}")
print(f"AUC Test:       {roc_auc_score(y_test, cal_probs_test):.4f}")
print(f"Brier (raw):        {brier_score_loss(y_val, raw_probs_val):.4f}")
print(f"Brier (calibrated): {brier_score_loss(y_val, cal_probs_val):.4f}")

# Calibration plot
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot([0,1], [0,1], 'k--', label='Perfect')
fp_raw, mp_raw = calibration_curve(y_val, raw_probs_val, n_bins=10)
ax.plot(mp_raw, fp_raw, 'b-o', label='Raw LR')
fp_cal, mp_cal = calibration_curve(y_val, cal_probs_val, n_bins=10)
ax.plot(mp_cal, fp_cal, 'r-o', label='Platt Scaled')
ax.set_title('Calibration Plot — Raw vs Platt Scaled')
ax.legend()
plt.tight_layout()
plt.show()

**Result:** Logistic regression scorecard AUC = 0.70. Platt scaling improved Brier score from 0.22 to 0.15.

In [ ]:
# Industry standard score scaling: BASE_SCORE=600, PDO=20
BASE_SCORE = 600
PDO = 20
BASE_ODDS = 1/0.213  # 1 / average_default_rate
FACTOR = PDO / np.log(2)
OFFSET = BASE_SCORE - (FACTOR * np.log(BASE_ODDS))

def prob_to_score(prob, factor=FACTOR, offset=OFFSET):
    prob = np.clip(prob, 1e-6, 1 - 1e-6)
    return round(offset + factor * np.log((1 - prob) / prob))

test_scores = np.array([prob_to_score(p) for p in calibrated_model.predict_proba(X_test_scaled)[:, 1]])

print(f"Min: {test_scores.min()}  Max: {test_scores.max()}  Mean: {test_scores.mean():.0f}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(test_scores[y_test == 0], bins=50, alpha=0.5, color='blue', label='Paid (Good)', density=True)
ax.hist(test_scores[y_test == 1], bins=50, alpha=0.5, color='red',  label='Default (Bad)', density=True)
ax.set_title('Credit Score Distribution — Default vs Paid')
ax.set_xlabel('Credit Score')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, classification_report, confusion_matrix, ConfusionMatrixDisplay

auc_test  = roc_auc_score(y_test, cal_probs_test)
fpr, tpr, _ = roc_curve(y_test, cal_probs_test)
ks_stat   = max(tpr - fpr)
gini_test = 2 * auc_test - 1

print(f"AUC: {auc_test:.4f}  |  Gini: {gini_test:.4f}  |  KS: {ks_stat:.4f}")

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, label=f'LR Scorecard (AUC={auc_test:.4f})')
ax.plot([0,1],[0,1],'k--')
ax.set_title('ROC Curve — Logistic Regression Scorecard')
ax.legend()
plt.tight_layout()
plt.show()

y_pred = (cal_probs_test >= 0.5).astype(int)
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred), display_labels=['Paid','Default']).plot(cmap='Blues')
plt.title('Confusion Matrix — Logistic Scorecard')
plt.show()
print(classification_report(y_test, y_pred, target_names=['Paid','Default']))

## 10. PD Modeling — LightGBM

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.frozen import FrozenEstimator

model_df_lgb = df_cleaned[feature_cols + [target_col]].dropna()
X_lgb = model_df_lgb[feature_cols]
y_lgb = model_df_lgb[target_col]

X_train_lgb, X_temp_lgb, y_train_lgb, y_temp_lgb = train_test_split(
    X_lgb, y_lgb, test_size=0.30, random_state=42, stratify=y_lgb)
X_val_lgb, X_test_lgb, y_val_lgb, y_test_lgb = train_test_split(
    X_temp_lgb, y_temp_lgb, test_size=0.50, random_state=42, stratify=y_temp_lgb)

print(f"Tuning LightGBM with {X_train_lgb.shape[0]:,} samples...")

param_grid = {
    'n_estimators': [300, 500], 'learning_rate': [0.05, 0.1],
    'max_depth': [4, 6], 'num_leaves': [31, 63],
    'min_child_samples': [100, 200], 'subsample': [0.8], 'colsample_bytree': [0.8],
}
base_lgb = lgb.LGBMClassifier(
    scale_pos_weight=len(y_train_lgb[y_train_lgb==0]) / len(y_train_lgb[y_train_lgb==1]),
    random_state=42, n_jobs=-1
)
search = RandomizedSearchCV(
    base_lgb, param_distributions=param_grid, n_iter=5,
    scoring='roc_auc', cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    random_state=42, n_jobs=-1, verbose=1
)
search.fit(X_train_lgb, y_train_lgb)

lgb_model = search.best_estimator_
calibrated_lgb = CalibratedClassifierCV(FrozenEstimator(lgb_model), method='sigmoid', cv='prefit')
calibrated_lgb.fit(X_val_lgb, y_val_lgb)

cal_probs_lgb = calibrated_lgb.predict_proba(X_test_lgb)[:, 1]
print(f"Final LightGBM Test AUC: {roc_auc_score(y_test_lgb, cal_probs_lgb):.4f}")

## 11. Survival Analysis — Kaplan-Meier

Survival analysis models *when* an event occurs, not just *whether* it does.

- **Kaplan-Meier:** Non-parametric — shows probability of surviving (not defaulting) to each month.
- **Censoring:** Active loans at data cutoff are censored — we know they survived *at least* N months.
- **Log-rank test:** Tests if survival curves from different groups are statistically different.

**Credit risk application:** KM confirms not just *if* but *when* default occurs. IFRS 9 lifetime PD estimation relies on this logic.

In [ ]:
!pip install lifelines --quiet

from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test

# Prepare survival dataset
survival_prep = df[['last_pymnt_d', 'issue_d']].copy()
survival_prep['last_pymnt_d'] = pd.to_datetime(survival_prep['last_pymnt_d'], format='mixed')
survival_prep['issue_d']      = pd.to_datetime(survival_prep['issue_d'],      format='mixed')
survival_prep['duration_months'] = (
    (survival_prep['last_pymnt_d'].dt.year  - survival_prep['issue_d'].dt.year)  * 12 +
    (survival_prep['last_pymnt_d'].dt.month - survival_prep['issue_d'].dt.month)
).clip(lower=1)

survival_df = df_cleaned[['default', 'fico_range_low', 'grade']].join(
    survival_prep['duration_months'], how='inner')
survival_df['purpose'] = df['purpose'].reindex(df_cleaned.index)
survival_df = survival_df.dropna()
print(f"Survival dataset: {survival_df.shape}, default rate: {survival_df['default'].mean():.4f}")

# KM by Grade
fig, ax = plt.subplots(figsize=(12, 7))
kmf = KaplanMeierFitter()
grade_colors = {'A':'#2ecc71','B':'#27ae60','C':'#f39c12','D':'#e67e22','E':'#e74c3c','F':'#c0392b','G':'#8e44ad'}
for grade, color in grade_colors.items():
    mask = survival_df['grade'] == grade
    if mask.sum() > 100:
        kmf.fit(survival_df.loc[mask,'duration_months'], survival_df.loc[mask,'default'],
                label=f'Grade {grade} (n={mask.sum():,})')
        kmf.plot_survival_function(ax=ax, color=color, ci_show=False)
ax.set_title('Kaplan-Meier Survival Curves by Loan Grade')
ax.set_xlabel('Months Since Origination')
ax.set_ylabel('Survival Probability')
ax.legend(loc='lower left')
ax.set_ylim(0.5, 1.0)
plt.tight_layout()
plt.show()

# KM by FICO Band
survival_df['fico_band'] = pd.cut(survival_df['fico_range_low'],
    bins=[0,650,670,690,720,750,850], labels=['<650','650-670','670-690','690-720','720-750','750+'])
fig, ax = plt.subplots(figsize=(12, 7))
for band, color in zip(['<650','650-670','670-690','690-720','720-750','750+'],
                        ['#e74c3c','#e67e22','#f39c12','#2ecc71','#27ae60','#1abc9c']):
    mask = survival_df['fico_band'] == band
    if mask.sum() > 100:
        kmf.fit(survival_df.loc[mask,'duration_months'], survival_df.loc[mask,'default'],
                label=f'FICO {band}')
        kmf.plot_survival_function(ax=ax, color=color, ci_show=False)
ax.set_title('Kaplan-Meier Survival Curves by FICO Band')
ax.set_ylim(0.5, 1.0)
ax.legend(loc='lower left')
plt.tight_layout()
plt.show()

# Multivariate log-rank test
print("\n--- Log-Rank Test: Are Grade Curves Different? ---")
results = multivariate_logrank_test(
    event_durations=survival_df['duration_months'],
    groups=survival_df['grade'],
    event_observed=survival_df['default']
)
results.print_summary()
print(f"P-value: {results.p_value:.6f}")

**Result:** Grade G reaches 50% cumulative default by month 20. Grade A survives to 82% at month 60. Log-rank p≈0 confirms differences are not random.

## 12. Cox Proportional Hazard Model

In [ ]:
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test

cox_features = [
    'int_rate', 'fico_range_low', 'dti', 'loan_amnt',
    'annual_inc', 'revol_util', 'loan_to_income',
    'mort_acc', 'has_pub_rec', 'grade_woe', 'home_ownership_woe'
]

cox_df = df_cleaned[cox_features + ['default']].copy()
cox_df['duration_months'] = survival_df['duration_months']
cox_df = cox_df.dropna()
cox_sample = cox_df.sample(n=100000, random_state=42)  # sample for speed

print("Fitting Cox model...")
cph = CoxPHFitter(penalizer=0.1)
cph.fit(cox_sample, duration_col='duration_months', event_col='default', show_progress=True)
cph.print_summary()

# Hazard ratio plot
fig, ax = plt.subplots(figsize=(10, 7))
cph.plot(ax=ax)
ax.set_title('Cox Model — Hazard Ratios with 95% CI')
ax.axvline(x=0, color='black', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# Proportional hazard assumption test
print("\n--- PH Assumption Test (Schoenfeld Residuals) ---")
ph_test = proportional_hazard_test(cph, cox_sample, time_transform='rank')
ph_test.print_summary()

# Sample borrower lifetime PD curves
sample_borrowers = pd.DataFrame({
    'int_rate':           [6.0,  11.0,  17.0,  25.0],
    'fico_range_low':     [780,   720,   670,   640],
    'dti':                [8.0,  15.0,  25.0,  35.0],
    'loan_amnt':          [5000, 12000, 20000, 30000],
    'annual_inc':         [90000, 65000, 45000, 35000],
    'revol_util':         [10.0,  35.0,  60.0,  80.0],
    'loan_to_income':     [0.06,  0.18,  0.44,  0.86],
    'mort_acc':           [3,     1,     0,     0],
    'has_pub_rec':        [0,     0,     0,     1],
    'grade_woe':          [1.355, 0.479, -0.559, -1.386],
    'home_ownership_woe': [0.182, 0.182, -0.192, -0.192],
})
labels = ['Very Low Risk (A)', 'Low Risk (B)', 'Medium Risk (D)', 'High Risk (G)']
colors = ['#2ecc71', '#f39c12', '#e67e22', '#e74c3c']
sf = cph.predict_survival_function(sample_borrowers)

fig, ax = plt.subplots(figsize=(12, 7))
for i, (label, color) in enumerate(zip(labels, colors)):
    ax.plot(sf.index, sf.iloc[:,i], label=label, color=color, linewidth=2)
ax.set_title('Cox Model — Individual Lifetime PD Curves')
ax.set_xlabel('Months Since Origination')
ax.set_ylabel('Survival Probability')
ax.legend()
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.show()

# 12/24/36m PD
print(f"\n{'Borrower':<30} {'12m PD':>10} {'24m PD':>10} {'36m PD':>10}")
for i, label in enumerate(labels):
    s = sf.iloc[:,i]
    pd12 = 1 - s.iloc[s.index.get_indexer([12], method='nearest')[0]]
    pd24 = 1 - s.iloc[s.index.get_indexer([24], method='nearest')[0]]
    pd36 = 1 - s.iloc[s.index.get_indexer([36], method='nearest')[0]]
    print(f"{label:<30} {pd12:>10.4f} {pd24:>10.4f} {pd36:>10.4f}")

**Note:** Six features violate the proportional hazard assumption. This is expected in credit risk — high-risk borrowers default early, leaving a stronger survivor population. For production use, a time-varying coefficient or stratified Cox model would address this.

## 13. Transition Matrix — Portfolio State Evolution

In [ ]:
def assign_state(status):
    if pd.isna(status): return np.nan
    status = str(status).strip()
    if status in ['Current', 'In Grace Period']:              return 0
    elif status in ['Late (16-30 days)']:                     return 1
    elif status in ['Late (31-60 days)']:                     return 2
    elif status in ['Late (61-90 days)', 'Late (91-120 days)']: return 3
    elif status in ['Charged Off', 'Default']:                return 4
    elif status in ['Fully Paid']:                            return 5
    else:                                                     return np.nan

def build_monthly_transitions(survival_data):
    """Reconstruct monthly delinquency transitions from loan history."""
    transitions = []
    for _, row in survival_data.iterrows():
        duration = max(int(row['duration_months']), 1)
        is_default = int(row['default'])
        if is_default == 1:
            for _ in range(max(duration - 4, 0)):
                transitions.append((0, 0))
            if duration >= 4:
                transitions.extend([(0,1),(1,2),(2,3),(3,4)])
            else:
                transitions.append((0, 4))
        else:
            for _ in range(max(duration - 1, 0)):
                transitions.append((0, 0))
            transitions.append((0, 5))
    return pd.DataFrame(transitions, columns=['from_state', 'to_state'])


n_states = 6
state_names = ['Current', '1-30 DPD', '31-60 DPD', '61-90 DPD', 'Default', 'Prepaid']

print("Sampling 50,000 loans for transition matrix...")
survival_sample = survival_df.sample(n=50000, random_state=42)
transitions_df = build_monthly_transitions(survival_sample)
print(f"Transitions generated: {len(transitions_df):,}")

count_matrix = np.zeros((n_states, n_states))
for _, row in transitions_df.iterrows():
    count_matrix[int(row['from_state'])][int(row['to_state'])] += 1

# Override DPD rows with industry cure/roll rates (data only captures final status, not monthly transitions)
count_matrix[1] = [60, 15, 20, 0, 0, 5]   # 1-30 DPD: mostly cure
count_matrix[2] = [30, 10, 15, 35, 5, 5]  # 31-60 DPD
count_matrix[3] = [10, 5, 10, 20, 50, 5]  # 61-90 DPD: mostly default
count_matrix[4][4] = 1  # Default — absorbing
count_matrix[5][5] = 1  # Prepaid — absorbing

transition_matrix = np.zeros((n_states, n_states))
for i in range(n_states):
    row_sum = count_matrix[i].sum()
    if row_sum > 0:
        transition_matrix[i] = count_matrix[i] / row_sum

tm_df = pd.DataFrame(transition_matrix, index=state_names, columns=state_names)
print("\n--- Transition Matrix ---")
print(tm_df.round(4))

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(tm_df, annot=True, fmt='.3f', cmap='RdYlGn', ax=ax, vmin=0, vmax=1, linewidths=0.5)
ax.set_title('Loan Delinquency Transition Matrix')
plt.tight_layout()
plt.show()

# Project 36 months forward
initial_portfolio = np.array([1_000_000, 0, 0, 0, 0, 0], dtype=float)
results, sv = [], initial_portfolio.copy()
for month in range(37):
    results.append({'month': month, 'Current': sv[0], '1-30 DPD': sv[1],
                    '31-60 DPD': sv[2], '61-90 DPD': sv[3],
                    'Default': sv[4], 'Prepaid': sv[5],
                    'Cum_Default_Rate': sv[4]/1_000_000})
    sv = sv @ transition_matrix
results_df = pd.DataFrame(results)

print(f"\n{'Month':<6} {'Current':>10} {'Default':>10} {'Prepaid':>10} {'Cum Default%':>14}")
for month in [0, 6, 12, 24, 36]:
    row = results_df[results_df['month']==month].iloc[0]
    print(f"{month:<6} {row['Current']:>10,.0f} {row['Default']:>10,.0f} {row['Prepaid']:>10,.0f} {row['Cum_Default_Rate']:>13.2%}")

# Grade-conditioned 12m and 36m PD
print("\n--- Grade-Conditioned Lifetime PD ---")
for grade in ['A','B','C','D','E','F','G']:
    grade_mask = survival_df['grade'] == grade
    if grade_mask.sum() < 100: continue
    gs = survival_df[grade_mask].sample(n=min(10000, grade_mask.sum()), random_state=42)
    gt = build_monthly_transitions(gs)
    gc = np.zeros((n_states, n_states))
    for _, row in gt.iterrows():
        gc[int(row['from_state'])][int(row['to_state'])] += 1
    gc[1]=[60,15,20,0,0,5]; gc[2]=[30,10,15,35,5,5]; gc[3]=[10,5,10,20,50,5]
    gc[4][4]=1; gc[5][5]=1
    gtm = np.zeros((n_states,n_states))
    for i in range(n_states):
        rs = gc[i].sum()
        if rs > 0: gtm[i] = gc[i]/rs
    sv12 = np.array([1e6,0,0,0,0,0],dtype=float)
    for _ in range(12): sv12 = sv12 @ gtm
    sv36 = np.array([1e6,0,0,0,0,0],dtype=float)
    for _ in range(36): sv36 = sv36 @ gtm
    print(f"Grade {grade} — 12m PD: {sv12[4]/1e6:.2%} | 36m PD: {sv36[4]/1e6:.2%}")

# Portfolio evolution plots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].stackplot(results_df['month'],
    results_df['Current'], results_df['1-30 DPD'],
    results_df['31-60 DPD'], results_df['61-90 DPD'],
    results_df['Default'], results_df['Prepaid'],
    labels=state_names,
    colors=['#2ecc71','#f39c12','#e67e22','#e74c3c','#c0392b','#3498db'], alpha=0.8)
axes[0].set_title('Portfolio State Evolution')
axes[0].legend(loc='upper right', fontsize=8)

axes[1].plot(results_df['month'], results_df['Cum_Default_Rate']*100, color='red', linewidth=2)
axes[1].axhline(y=results_df.iloc[12]['Cum_Default_Rate']*100, color='orange', linestyle='--',
                label=f"12m: {results_df.iloc[12]['Cum_Default_Rate']:.1%}")
axes[1].axhline(y=results_df.iloc[36]['Cum_Default_Rate']*100, color='red', linestyle='--',
                label=f"36m: {results_df.iloc[36]['Cum_Default_Rate']:.1%}")
axes[1].set_title('Cumulative Portfolio Default Rate')
axes[1].legend()
plt.tight_layout()
plt.show()

**Result:** Grade G 36m PD (10.6%) is 17x higher than Grade A (0.6%), consistent with KM and Cox findings. Absolute rates are understated because the dataset only records final loan status, not monthly payment history.

## 14. Model Comparison — Cox vs Transition Matrix

In [ ]:
print(f"\n{'Borrower':<25} {'Cox 12m':>10} {'Cox 36m':>10} {'TM 12m':>10} {'TM 36m':>10}")
print("-" * 65)

cox_12 = {'A': 0.0316, 'B': 0.0669, 'G': 0.3560}
cox_36 = {'A': 0.1082, 'B': 0.2185, 'G': 0.7914}

for grade in ['A', 'B', 'G']:
    gs = survival_df[survival_df['grade']==grade].sample(n=min(10000, (survival_df['grade']==grade).sum()), random_state=42)
    gt = build_monthly_transitions(gs)
    gc = np.zeros((n_states,n_states))
    for _, row in gt.iterrows(): gc[int(row['from_state'])][int(row['to_state'])] += 1
    gc[1]=[60,15,20,0,0,5]; gc[2]=[30,10,15,35,5,5]; gc[3]=[10,5,10,20,50,5]
    gc[4][4]=1; gc[5][5]=1
    gtm = np.zeros((n_states,n_states))
    for i in range(n_states):
        rs = gc[i].sum()
        if rs > 0: gtm[i] = gc[i]/rs
    sv12 = np.array([1e6,0,0,0,0,0],dtype=float)
    for _ in range(12): sv12 = sv12 @ gtm
    sv36 = np.array([1e6,0,0,0,0,0],dtype=float)
    for _ in range(36): sv36 = sv36 @ gtm
    print(f"Grade {grade:<20} {cox_12[grade]:>10.2%} {cox_36[grade]:>10.2%} {sv12[4]/1e6:>10.2%} {sv36[4]/1e6:>10.2%}")

print("""
--- Where they diverge and why ---

Cox: individual level, handles censoring, captures non-linear interactions, higher PD estimates.
TM:  portfolio level, based on reconstructed data, simpler, lower absolute PD.

Both agree on ORDERING (Grade G >> Grade A) and SHAPE (PD increases over time).
Diverge on MAGNITUDE — Cox 36m Grade G = 79% vs TM 36m Grade G = 10%.
Gap = data quality difference (no monthly payment history), not methodology flaw.
""")

## 15. Loss Given Default (LGD) Modeling

In [ ]:
print("Filtering to defaulted loans...")
defaulted_df = df[df['loan_status'].isin(['Charged Off', 'Default'])].copy()
print(f"Defaulted loans: {len(defaulted_df):,} ({len(defaulted_df)/len(df):.2%} of portfolio)")

# Basic LGD: (loan_amnt - total_pymnt) / loan_amnt
defaulted_df['lgd'] = (
    (defaulted_df['loan_amnt'] - defaulted_df['total_pymnt']) / defaulted_df['loan_amnt']
).clip(0, 1)

print(f"Mean LGD:   {defaulted_df['lgd'].mean():.4f}")
print(f"Median LGD: {defaulted_df['lgd'].median():.4f}")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].hist(defaulted_df['lgd'], bins=50, color='#e74c3c', edgecolor='white', alpha=0.8)
axes[0].axvline(defaulted_df['lgd'].mean(), color='black', linestyle='--',
                label=f"Mean: {defaulted_df['lgd'].mean():.3f}")
axes[0].set_title('LGD Distribution — Defaulted Loans'); axes[0].legend()
sns.kdeplot(defaulted_df['lgd'], ax=axes[1], color='#e74c3c', fill=True, alpha=0.4)
axes[1].set_title('LGD Density Plot')
plt.tight_layout(); plt.show()

# By grade
grade_lgd = defaulted_df.groupby('grade')['lgd'].mean().sort_index()
grade_lgd.plot(kind='bar', color='#e74c3c', alpha=0.8, title='Mean LGD by Grade')
plt.tight_layout(); plt.show()

In [ ]:
# Refined LGD: also includes post-default collections, net of collection fees
defaulted_df['total_recovered_refined'] = (
    defaulted_df['total_pymnt'] + defaulted_df['recoveries'] - defaulted_df['collection_recovery_fee']
)
defaulted_df['lgd_refined'] = (
    (defaulted_df['loan_amnt'] - defaulted_df['total_recovered_refined']) / defaulted_df['loan_amnt']
).clip(0, 1)

print(f"LGD (basic):   {defaulted_df['lgd'].mean():.4f}")
print(f"LGD (refined): {defaulted_df['lgd_refined'].mean():.4f}")

defaulted_df['lgd'] = defaulted_df['lgd_refined']  # use refined going forward
print(f"Using refined LGD: mean={defaulted_df['lgd'].mean():.4f}")

**LGD basic = 0.4672** — only payments made while loan was active. **LGD refined = 0.4137** — also includes post-default collections minus collection fees.

Lending Club LGD is NOT the typical bimodal spike-at-1.0 seen in mortgage data. Left spike near 0 and broad right hump peaking ~0.6 — personal loans have no collateral but borrowers typically make partial payments before defaulting.

### Two-Stage LGD Model

In [ ]:
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

defaulted_df['lgd'] = defaulted_df['lgd_refined']

lgd_features = [
    'int_rate', 'fico_range_low', 'dti', 'loan_amnt',
    'annual_inc', 'revol_util', 'loan_to_income',
    'mort_acc', 'has_pub_rec', 'grade_woe', 'home_ownership_woe'
]

lgd_df = defaulted_df[['loan_amnt', 'lgd', 'grade', 'purpose']].copy()
for feat in lgd_features:
    if feat in defaulted_df.columns: lgd_df[feat] = defaulted_df[feat]
    elif feat in df_cleaned.columns: lgd_df[feat] = df_cleaned[feat].reindex(defaulted_df.index)
lgd_df = lgd_df.dropna(subset=lgd_features + ['lgd'])
lgd_df['stage1_target'] = (lgd_df['lgd'] >= 0.10).astype(int)

X_lgd = lgd_df[lgd_features]
y1 = lgd_df['stage1_target']
y_lgd_vals = lgd_df['lgd']

X_tr, X_te, y1_tr, y1_te, yl_tr, yl_te = train_test_split(
    X_lgd, y1, y_lgd_vals, test_size=0.2, random_state=42, stratify=y1)

scaler_lgd = StandardScaler()
X_tr_s = scaler_lgd.fit_transform(X_tr)
X_te_s = scaler_lgd.transform(X_te)

# Stage 1 — logistic
s1 = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
s1.fit(X_tr_s, y1_tr)
print(f"Stage 1 AUC: {roc_auc_score(y1_te, s1.predict_proba(X_te_s)[:,1]):.4f}")

# Stage 2 — linear on loss group
loss_mask = y1_tr == 1
s2 = LinearRegression()
s2.fit(X_tr_s[loss_mask], yl_tr[loss_mask])

def predict_lgd(X_input):
    Xs = scaler_lgd.transform(X_input)
    prob = s1.predict_proba(Xs)[:,1]
    lgd_loss = np.clip(s2.predict(Xs), 0, 1)
    return prob * lgd_loss + (1 - prob) * 0.05

preds_v1 = predict_lgd(X_te)
rmse_v1 = np.sqrt(mean_squared_error(yl_te, preds_v1))
print(f"Two-Stage LGD V1 RMSE: {rmse_v1:.4f}")
print(f"Mean Predicted: {preds_v1.mean():.4f} | Mean Actual: {yl_te.mean():.4f}")

In [ ]:
# V2: add behavioral features (loan age at default, payment ratio)
defaulted_df['loan_age_at_default'] = survival_df['duration_months'].reindex(defaulted_df.index)
defaulted_df['payment_ratio'] = (defaulted_df['total_pymnt'] / defaulted_df['loan_amnt']).clip(0, 1)
if 'installment' in defaulted_df.columns:
    defaulted_df['months_paid_ratio'] = (defaulted_df['total_pymnt'] / defaulted_df['installment']).clip(0, None)
defaulted_df['recoveries_ratio'] = (defaulted_df['recoveries'] / defaulted_df['loan_amnt']).clip(0, 1)

print("--- Correlation with LGD ---")
for feat in ['loan_age_at_default', 'payment_ratio', 'recoveries_ratio', 'int_rate', 'grade_woe']:
    if feat in defaulted_df.columns:
        corr = defaulted_df[[feat,'lgd']].dropna().corr().iloc[0,1]
        print(f"  {feat:<30} {corr:>8.4f}")

lgd_features_v2 = lgd_features + ['loan_age_at_default', 'payment_ratio', 'recoveries_ratio']
if 'months_paid_ratio' in defaulted_df.columns:
    lgd_features_v2.append('months_paid_ratio')

lgd_df_v2 = defaulted_df[['loan_amnt','lgd','grade']].copy()
for feat in lgd_features_v2:
    if feat in defaulted_df.columns: lgd_df_v2[feat] = defaulted_df[feat]
    elif feat in df_cleaned.columns: lgd_df_v2[feat] = df_cleaned[feat].reindex(defaulted_df.index)
lgd_df_v2 = lgd_df_v2.dropna(subset=lgd_features_v2+['lgd'])
lgd_df_v2['stage1_target'] = (lgd_df_v2['lgd'] >= 0.10).astype(int)

X2 = lgd_df_v2[lgd_features_v2]
y1_2 = lgd_df_v2['stage1_target']
yl2 = lgd_df_v2['lgd']

X2_tr, X2_te, y1_2_tr, y1_2_te, yl2_tr, yl2_te = train_test_split(
    X2, y1_2, yl2, test_size=0.2, random_state=42, stratify=y1_2)

scaler_v2 = StandardScaler()
X2_tr_s = scaler_v2.fit_transform(X2_tr)
X2_te_s = scaler_v2.transform(X2_te)

s1_v2 = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
s1_v2.fit(X2_tr_s, y1_2_tr)

lm_v2 = y1_2_tr == 1
s2_v2 = LinearRegression()
s2_v2.fit(X2_tr_s[lm_v2], yl2_tr[lm_v2])

def predict_lgd_v2(X_input):
    Xs = scaler_v2.transform(X_input)
    prob = s1_v2.predict_proba(Xs)[:,1]
    lgd_loss = np.clip(s2_v2.predict(Xs), 0, 1)
    return prob * lgd_loss + (1 - prob) * 0.05

preds_v2 = predict_lgd_v2(X2_te)
PORTFOLIO_LGD = preds_v2.mean()
rmse_v2 = np.sqrt(mean_squared_error(yl2_te, preds_v2))
print(f"\nTwo-Stage LGD V2 RMSE: {rmse_v2:.4f}")
print(f"Portfolio LGD: {PORTFOLIO_LGD:.4f}")

# Grade comparison
res_v2 = X2_te.copy()
res_v2['actual'] = yl2_te.values
res_v2['predicted'] = preds_v2
res_v2['grade'] = lgd_df_v2.loc[X2_te.index, 'grade']
print("\n--- LGD by Grade: Actual vs Predicted ---")
print(res_v2.groupby('grade').agg(actual=('actual','mean'), predicted=('predicted','mean'), count=('actual','count')).round(4))

**Result:** Two-stage LGD V2 achieves RMSE=0.019 after adding behavioral features. Payment ratio (correlation -0.974 with LGD) and loan age at default (-0.796) dominate — LGD for unsecured loans is driven by borrower payment behavior, not origination characteristics. Grade-level predictions within 0.006 of actuals. Portfolio LGD = 0.41.

## 16. EAD + Expected Loss

In [ ]:
# 1. VECTORIZED EAD — amortization formula
# EAD = remaining balance = loan_amnt × [(1+r)^n - (1+r)^t] / [(1+r)^n - 1]
print("Computing EAD...")
if 'term' in df.columns:
    all_term = df.loc[df_cleaned.index, 'term'].str.extract(r'(\d+)').astype(float).squeeze()
else:
    all_term = pd.Series(36, index=df_cleaned.index)

all_duration = survival_df['duration_months'].reindex(df_cleaned.index).fillna(18)
all_int_rate = df_cleaned['int_rate']
all_loan_amnt = df.loc[df_cleaned.index, 'loan_amnt']

mr = all_int_rate / 12 / 100
n = all_term
t = all_duration.clip(upper=all_term)
num = (1+mr)**n - (1+mr)**t
den = (1+mr)**n - 1
ead_fraction = np.where(den==0, np.maximum(1-t/n,0), (num/den).clip(0,1))
all_ead_fraction = pd.Series(ead_fraction, index=df_cleaned.index)
all_ead = all_loan_amnt * all_ead_fraction

# 2. PD — calibrated LightGBM
print("Computing PD...")
all_pd = pd.Series(calibrated_lgb.predict_proba(df_cleaned[feature_cols])[:,1], index=df_cleaned.index)

# 3. LGD — two-stage model with leakage-free inputs
print("Computing LGD...")
lgd_input = df_cleaned[[c for c in lgd_features_v2 if c in df_cleaned.columns]].copy()
lgd_input['loan_age_at_default'] = all_duration
lgd_input['payment_ratio'] = (1 - all_ead_fraction).clip(0, 1)  # amortization-implied, no leakage
lgd_input['recoveries_ratio'] = 0  # 0 for performing loans
lgd_input['months_paid_ratio'] = all_duration.clip(lower=0)
lgd_input = lgd_input[lgd_features_v2].fillna(lgd_input.median())
all_lgd = pd.Series(predict_lgd_v2(lgd_input), index=df_cleaned.index)

# Blend: PD-weighted average of model LGD and empirical mean (corrects distribution shift)
EMPIRICAL_MEAN_LGD = defaulted_df['lgd'].mean()
all_lgd = all_pd * all_lgd + (1 - all_pd) * EMPIRICAL_MEAN_LGD
print(f"Mean LGD after blend: {all_lgd.mean():.4f}")

# 4. EXPECTED LOSS = PD × LGD × EAD
el_df = pd.DataFrame({
    'loan_amnt': all_loan_amnt, 'pd': all_pd, 'lgd': all_lgd,
    'ead': all_ead, 'ead_fraction': all_ead_fraction,
    'grade': df.loc[df_cleaned.index, 'grade']
}, index=df_cleaned.index)
el_df['expected_loss'] = el_df['pd'] * el_df['lgd'] * el_df['ead']
el_df['el_rate'] = el_df['expected_loss'] / el_df['loan_amnt']

print(f"\n--- Portfolio Expected Loss Summary ---")
print(f"Total Exposure: ${el_df['loan_amnt'].sum():,.0f}")
print(f"Total EL:       ${el_df['expected_loss'].sum():,.0f}")
print(f"EL Rate:        {el_df['expected_loss'].sum()/el_df['loan_amnt'].sum():.2%}")

grade_summary = el_df.groupby('grade').agg(
    mean_pd=('pd','mean'), mean_lgd=('lgd','mean'),
    mean_ead_frac=('ead_fraction','mean'), el_rate=('el_rate','mean'),
    total_el=('expected_loss','sum')
).round(4)
print(f"\n--- EL by Grade ---")
print(grade_summary)

grade_summary['el_rate'].plot(kind='bar', color='darkred', alpha=0.7, title='Expected Loss Rate by Grade')
plt.ylabel('EL Rate')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# EL by Vintage
el_df['issue_year'] = pd.to_datetime(df.loc[df_cleaned.index,'issue_d'], format='mixed').dt.year
print("--- EL by Vintage ---")
print(el_df.groupby('issue_year').agg(
    mean_pd=('pd','mean'), mean_lgd=('lgd','mean'),
    el_rate=('el_rate','mean'), total_el=('expected_loss','sum'), count=('loan_amnt','count')
).round(4))

# EL by Purpose
el_df['purpose'] = df['purpose'].reindex(df_cleaned.index)
print("\n--- EL by Purpose ---")
print(el_df.groupby('purpose').agg(
    el_rate=('el_rate','mean'), total_el=('expected_loss','sum'), count=('loan_amnt','count')
).round(4).sort_values('el_rate', ascending=False))

# EL Distribution + Tail
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].hist(el_df['el_rate'], bins=100, color='#e74c3c', alpha=0.8)
axes[0].axvline(el_df['el_rate'].mean(), color='black', linestyle='--',
                label=f"Mean: {el_df['el_rate'].mean():.4f}")
axes[0].axvline(el_df['el_rate'].quantile(0.95), color='gold', linestyle='--',
                label=f"95th: {el_df['el_rate'].quantile(0.95):.4f}")
axes[0].set_title('EL Rate Distribution — Full Portfolio')
axes[0].legend()

tail = el_df[el_df['el_rate'] >= el_df['el_rate'].quantile(0.95)]
axes[1].hist(tail['el_rate'], bins=50, color='#c0392b', alpha=0.8)
axes[1].set_title(f'EL Tail (Top 5%) — {len(tail):,} loans, ${tail["expected_loss"].sum():,.0f} total EL')
plt.tight_layout()
plt.show()

print(f"95th pct EL rate: {el_df['el_rate'].quantile(0.95):.4f}")
print(f"Top 5% loans: {tail['expected_loss'].sum()/el_df['expected_loss'].sum():.1%} of total EL")

### Reading the Expected Loss Number

**Portfolio EL = $1.34B on $19.4B exposure (6.89% EL rate)**

Three modeling choices drive this number:

- **PD** — Calibrated LightGBM. Range: 6% (Grade A) to 49% (Grade G).
- **LGD** — Two-stage model blended with empirical mean (0.41) weighted by PD. Blend corrects for distribution shift between defaulter training population and performing loan prediction population.
- **EAD** — Amortization formula. `payment_ratio` uses scheduled paydown (1 − EAD fraction) rather than observed `total_pymnt` to avoid leakage on non-defaulted loans.

Grade G EL rate (24.65%) is **22x Grade A (1.09%)** — driven by simultaneously higher PD, LGD, and EAD. The multiplicative effect of all three components explains the dramatic grade spread. This EL figure represents the minimum loan loss reserve under IFRS 9 Stage 1 provisioning.

# Part II — Macro Stress, Regulatory Capital, Portfolio Simulation & Structural Models

## 17. Macro Integration — FRED Data & Time-Varying Cox Model

In [ ]:
# ============================================================
# CELL 1
# ============================================================
!pip install fredapi --quiet

import os
from fredapi import Fred

FRED_API_KEY = os.environ.get("FRED_API_KEY")  # get a free key at https://fred.stlouisfed.org/docs/api/api_key.html
fred = Fred(api_key=FRED_API_KEY)

In [ ]:
# ============================================================
# CELL 2 — Pull macro series from FRED
# ============================================================
macro_series = {
    'unemployment_rate': 'UNRATE',
    'gdp_growth':        'A191RL1Q225SBEA',   # quarterly, will be forward-filled to monthly
    'treasury_10y':      'DGS10',
    'case_shiller_hpi':  'CSUSHPISA',
    'vix':               'VIXCLS',
}

macro_raw = {}
for name, code_ in macro_series.items():
    s = fred.get_series(code_)
    s.index = pd.to_datetime(s.index)
    macro_raw[name] = s
    print(f"{name:<20} ({code_}): {s.index.min().date()} to {s.index.max().date()}, {len(s)} obs")

In [ ]:
# ============================================================
# CELL 3 — Build a clean monthly macro panel
# ============================================================
macro_monthly = pd.DataFrame(index=pd.date_range('2006-01-01', '2019-06-01', freq='MS'))

for name, s in macro_raw.items():
    monthly = s.resample('MS').mean()
    monthly = monthly.reindex(macro_monthly.index).ffill()
    macro_monthly[name] = monthly

macro_monthly = macro_monthly.dropna()
print(f"Macro panel shape: {macro_monthly.shape}")
macro_monthly.tail()

In [ ]:
# ============================================================
# CELL 4 — Visual sanity check, recession period shaded red
# ============================================================
fig, axes = plt.subplots(len(macro_series), 1, figsize=(12, 14), sharex=True)
for ax, name in zip(axes, macro_series.keys()):
    ax.plot(macro_monthly.index, macro_monthly[name], color='#2c3e50')
    ax.axvspan(pd.Timestamp('2007-12-01'), pd.Timestamp('2009-06-01'), color='red', alpha=0.15)
    ax.set_title(name)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 5 — Macro value at origination (static building block)
# ============================================================
issue_dates = pd.to_datetime(df_cleaned['issue_d'], format='mixed')
issue_month_start = issue_dates.values.astype('datetime64[M]').astype('datetime64[ns]')

macro_at_origination = macro_monthly.reindex(issue_month_start).reset_index(drop=True)
macro_at_origination.index = df_cleaned.index

for col in macro_monthly.columns:
    df_cleaned[f'{col}_origination'] = macro_at_origination[col]

print("Macro-at-origination merge check:")
df_cleaned[[f'{c}_origination' for c in macro_monthly.columns]].describe()

In [ ]:
# ============================================================
# CELL 6 — Build the long-format (time-varying) sample
# ============================================================
from lifelines import CoxTimeVaryingFitter

N_SAMPLE = 30000  # start here; raise if runtime allows

tv_base = df_cleaned[cox_features + ['default']].copy()
tv_base['duration_months'] = survival_df['duration_months'].reindex(tv_base.index)
tv_base['issue_d'] = issue_dates.reindex(tv_base.index)
tv_base = tv_base.dropna(subset=cox_features + ['default', 'duration_months', 'issue_d'])

tv_sample = tv_base.sample(n=min(N_SAMPLE, len(tv_base)), random_state=42).reset_index(drop=True)
tv_sample['id'] = tv_sample.index

print(f"Building long-format panel for {len(tv_sample):,} loans...")

In [ ]:
# ============================================================
# CELL 7 — Reshape to long format (this is the slow cell)
# ============================================================
macro_lookup = macro_monthly['unemployment_rate']

long_rows = []
for _, row in tv_sample.iterrows():
    duration = int(row['duration_months'])
    issue = row['issue_d'].to_period('M').to_timestamp()
    for month in range(duration):
        current_date = (issue + pd.DateOffset(months=month)).to_period('M').to_timestamp()
        unrate = macro_lookup.get(current_date, np.nan)
        long_rows.append({
            'id': row['id'],
            'start': month,
            'stop': month + 1,
            'event': 1 if (month == duration - 1 and row['default'] == 1) else 0,
            'unemployment_rate': unrate,
            **{f: row[f] for f in cox_features}
        })

long_df = pd.DataFrame(long_rows).dropna()
print(f"Long-format rows: {len(long_df):,}  (from {len(tv_sample):,} loans)")

In [ ]:
# ============================================================
# CELL 8 — Fit the time-varying Cox model
# ============================================================
ctv = CoxTimeVaryingFitter(penalizer=0.1)
ctv.fit(long_df, id_col='id', start_col='start', stop_col='stop', event_col='event',
        show_progress=True)
ctv.print_summary()

In [ ]:
# ============================================================
# CELL 9 — Extract the unemployment hazard ratio
# ============================================================
unrate_coef = ctv.params_['unemployment_rate']
unrate_hr = np.exp(unrate_coef)

print(f"Unemployment coefficient (beta): {unrate_coef:.4f}")
print(f"Unemployment hazard ratio:       {unrate_hr:.4f}")
print(f"\nInterpretation: each 1-point rise in unemployment multiplies instantaneous "
      f"default hazard by {unrate_hr:.3f}x, holding borrower characteristics fixed.")

ctv.plot()
plt.title('Time-Varying Cox — Hazard Ratios incl. Macro')
plt.tight_layout()
plt.show()

**First attempt result:** the naive time-varying Cox model returns an unemployment hazard ratio of 0.96 — i.e. *higher* unemployment slightly *lowers* default hazard, holding borrower characteristics fixed. That's directionally implausible, so before trusting it, the next cell checks whether it's actually picking up unemployment's effect or just absorbing something else (origination-vintage drift).

In [ ]:
# ============================================================
# FIX — Add a vintage-year control to separate seasoning/underwriting
# drift from unemployment's actual effect
# ============================================================

# Re-use the same tv_sample from before (or re-sample if you want a fresh draw)
tv_sample['issue_year'] = tv_sample['issue_d'].dt.year

# Use a continuous "months since first loan" trend rather than a categorical
# year — this absorbs the steady vintage/underwriting drift as a straight-line
# trend, leaving unemployment's month-to-month wiggles for its own coefficient
# to capture.
tv_sample['vintage_trend'] = (
    (tv_sample['issue_year'] - tv_sample['issue_year'].min()) * 12 +
    (tv_sample['issue_d'].dt.month - 1)
)

print("Vintage trend range:", tv_sample['vintage_trend'].min(), "to", tv_sample['vintage_trend'].max())

In [ ]:
# ============================================================
# Rebuild long_rows with vintage_trend included as a covariate
# ============================================================
macro_lookup = macro_monthly['unemployment_rate']

long_rows = []
for _, row in tv_sample.iterrows():
    duration = int(row['duration_months'])
    issue = row['issue_d'].to_period('M').to_timestamp()
    for month in range(duration):
        current_date = (issue + pd.DateOffset(months=month)).to_period('M').to_timestamp()
        unrate = macro_lookup.get(current_date, np.nan)
        long_rows.append({
            'id': row['id'],
            'start': month,
            'stop': month + 1,
            'event': 1 if (month == duration - 1 and row['default'] == 1) else 0,
            'unemployment_rate': unrate,
            'vintage_trend': row['vintage_trend'],   # NEW — absorbs seasoning/underwriting drift
            **{f: row[f] for f in cox_features}
        })

long_df = pd.DataFrame(long_rows).dropna()
print(f"Long-format rows: {len(long_df):,}  (from {len(tv_sample):,} loans)")

In [ ]:
# ============================================================
# Refit the time-varying Cox model WITH the vintage control
# ============================================================
from lifelines import CoxTimeVaryingFitter

ctv_v2 = CoxTimeVaryingFitter(penalizer=0.1)
ctv_v2.fit(long_df, id_col='id', start_col='start', stop_col='stop', event_col='event',
           show_progress=True)
ctv_v2.print_summary()

In [ ]:
# ============================================================
# Extract and compare the corrected unemployment hazard ratio
# ============================================================
unrate_coef_v2 = ctv_v2.params_['unemployment_rate']
unrate_hr_v2 = np.exp(unrate_coef_v2)

vintage_coef = ctv_v2.params_['vintage_trend']
vintage_hr = np.exp(vintage_coef)

print(f"--- BEFORE (no vintage control) ---")
print(f"Unemployment hazard ratio: {unrate_hr:.4f}")
print()
print(f"--- AFTER (with vintage control) ---")
print(f"Unemployment hazard ratio: {unrate_hr_v2:.4f}")
print(f"Vintage trend hazard ratio (per month of platform age): {vintage_hr:.4f}")
print()
print(f"Interpretation: each 1-point rise in unemployment now multiplies default "
      f"hazard by {unrate_hr_v2:.3f}x, holding borrower characteristics AND vintage "
      f"drift fixed.")
print(f"The vintage_trend coefficient absorbs the steady seasoning/underwriting-drift "
      f"effect separately — {vintage_hr:.4f}x per additional month of platform age.")

ctv_v2.plot()
plt.title('Time-Varying Cox WITH Vintage Control — Hazard Ratios')
plt.tight_layout()
plt.show()

# ============================================================
# Use this corrected value going forward for stress testing
# ============================================================
unrate_hr = unrate_hr_v2   # overwrite — this is now the value the afternoon
                            # stress-testing block will use
print(f"unrate_hr updated to corrected value: {unrate_hr:.4f}")

### Diagnosing the identification problem

In [ ]:
# ============================================================
# DIAGNOSTIC 1 — How much does unemployment_rate actually vary
# within the sample the model is using?
# ============================================================
print("--- Unemployment rate distribution in long_df ---")
print(long_df['unemployment_rate'].describe())

print("\n--- Unemployment rate by issue year (in the sample) ---")
year_map = tv_sample.set_index('id')['issue_year']
long_df['issue_year'] = long_df['id'].map(year_map)
print(long_df.groupby('issue_year')['unemployment_rate'].agg(['mean', 'min', 'max', 'std']))

# If std is tiny and min/max range is narrow, the model has very little
# signal to learn an unemployment effect from at all.

In [ ]:
# ============================================================
# DIAGNOSTIC 2 — Raw relationship: unemployment vs default,
# no model, just direct comparison
# ============================================================

# Bucket loan-months by unemployment level, look at raw event (default) rate
long_df['unrate_bucket'] = pd.cut(long_df['unemployment_rate'],
                                    bins=[0, 4, 5, 6, 7, 8, 100],
                                    labels=['<4%', '4-5%', '5-6%', '6-7%', '7-8%', '8%+'])

print("--- Raw event rate by unemployment bucket (loan-month level) ---")
print(long_df.groupby('unrate_bucket').agg(
    event_rate=('event', 'mean'),
    n_months=('event', 'count')
))

# Also check: does this just mirror issue_year, i.e. is unrate_bucket basically
# identical to issue_year in disguise?
print("\n--- Cross-tab: issue_year vs unrate_bucket (loan-month level) ---")
print(pd.crosstab(long_df['issue_year'], long_df['unrate_bucket']))

In [ ]:
# ============================================================
# DIAGNOSTIC 3 — Is unemployment_rate collinear with int_rate or grade_woe?
# (i.e. is the model "stealing" unemployment's signal via a correlated feature)
# ============================================================

corr_check_cols = ['unemployment_rate', 'int_rate', 'grade_woe', 'vintage_trend']
print("--- Correlation matrix among these covariates (loan-month level) ---")
print(long_df[corr_check_cols].corr().round(3))

# A high correlation (e.g. |corr| > 0.5) between unemployment_rate and
# int_rate or grade_woe would mean the model can't cleanly tell them apart —
# whichever one "wins" the regression is partly arbitrary.

### Limitation and Resolution — Macro Sensitivity Could Not Be Reliably Estimated

A time-varying Cox model was fit to estimate the sensitivity of default hazard to the unemployment rate. The fitted hazard ratio (0.96) was directionally implausible — diagnostics showed origination vintage and unemployment rate are correlated at r = -0.755 in this sample, meaning the dataset lacks sufficient independent variation between calendar time and macro conditions to identify unemployment's effect separately from vintage/underwriting-quality drift (Lending Club's loan volume and underwriting standards changed substantially between 2012-2017, the same window over which unemployment fell steadily post-recession). Adding a vintage control did not resolve this, confirming the two variables are too collinear to disentangle in this sample.
Given this identification problem, the stress-testing exercise below uses an assumed hazard ratio of 1.10–1.15 per 1-point unemployment increase, consistent with the direction and approximate magnitude found in published consumer-credit research (e.g., Gerardi et al., Federal Reserve Bank of Atlanta, find unemployment raises default probability by 5-13 percentage points in mortgage data), rather than the dataset's own (unreliable) fitted coefficient. Notably, the Federal Reserve has also documented cases (COVID-19) where this historical relationship temporarily decoupled due to policy interventions — a reminder that even well-identified macro sensitivities are not guaranteed to hold under all conditions.

## 18. Macro Stress Scenarios

In [ ]:
# ============================================================
# CELL 1 — Document the limitation, set the assumed hazard ratio
# ============================================================

# The dataset's own fitted unemployment hazard ratio (0.96) was found to be
# unreliable due to an identification problem: origination vintage and
# unemployment rate are correlated at r=-0.755 in this sample, meaning there
# isn't enough independent variation to separate "calendar time drift" from
# "macro effect." Diagnostics (crosstab of issue_year x unemployment bucket)
# confirmed near-zero overlap between vintages across unemployment levels.
#
# Rather than use the dataset's own unreliable coefficient, this stress test
# uses an assumed hazard ratio consistent with published consumer-credit
# research (Gerardi et al., Federal Reserve Bank of Atlanta — unemployment
# raises default probability by 5-13 percentage points in mortgage data).

ASSUMED_UNRATE_HR = 1.12   # midpoint of a defensible 1.10-1.15 literature-based range

print(f"Using assumed unemployment hazard ratio: {ASSUMED_UNRATE_HR}")
print(f"(Dataset's own fitted value was {unrate_hr:.4f} — rejected as unreliable; see writeup.)")

unrate_hr = ASSUMED_UNRATE_HR  # overwrite so downstream stress-test cells use this

In [ ]:
# ============================================================
# CELL 2 — Define the three stress scenarios
# ============================================================
scenarios = {
    'Baseline':         {'unemployment_delta': 0.0},
    'Adverse':          {'unemployment_delta': 3.0},
    'Severely Adverse':  {'unemployment_delta': 6.0},  # roughly 2008-magnitude shock
}

for name, shock in scenarios.items():
    print(f"{name:<18} unemployment +{shock['unemployment_delta']:.1f} pts")

In [ ]:
# ============================================================
# CELL 3 — Apply the hazard-ratio multiplier to PD, recompute portfolio EL
# ============================================================
baseline_pd = all_pd.copy()  # from Week 1's EAD + Expected Loss section

stress_results = {}
for name, shock in scenarios.items():
    multiplier = unrate_hr ** shock['unemployment_delta']
    stressed_pd = (baseline_pd * multiplier).clip(0, 1)
    stressed_el = stressed_pd * all_lgd * all_ead
    stress_results[name] = {
        'pd_multiplier': multiplier,
        'mean_pd': stressed_pd.mean(),
        'total_el': stressed_el.sum(),
        'el_rate': stressed_el.sum() / all_loan_amnt.sum(),
    }

stress_summary = pd.DataFrame(stress_results).T
stress_summary['el_increase_vs_baseline'] = (
    stress_summary['total_el'] / stress_summary.loc['Baseline', 'total_el'] - 1
)
print(stress_summary.round(4))

In [ ]:
# ============================================================
# CELL 4 — Plot EL rate under each scenario
# ============================================================
fig, ax = plt.subplots(figsize=(10, 6))
stress_summary['el_rate'].plot(kind='bar', color=['#2ecc71', '#f39c12', '#e74c3c'], ax=ax)
ax.set_title('Portfolio EL Rate Under Stress Scenarios\n(assumed hazard ratio = 1.12/pt, literature-based)')
ax.set_ylabel('EL Rate')
for i, v in enumerate(stress_summary['el_rate']):
    ax.text(i, v + 0.002, f'{v:.2%}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

### Transition Matrix by Economic Regime

In [ ]:
# ============================================================
# CELL 5 — Transition matrix split: expansion vs. recession regime
# (kept from the original plan — separate exercise, not affected by the
# hazard-ratio issue above, but carries its own small-sample caveat)
# ============================================================
RECESSION_START = pd.Timestamp('2007-12-01')
RECESSION_END   = pd.Timestamp('2009-06-01')

survival_df_regime = survival_df.copy()
survival_df_regime['issue_d'] = issue_dates.reindex(survival_df_regime.index)
survival_df_regime['regime'] = np.where(
    (survival_df_regime['issue_d'] >= RECESSION_START) & (survival_df_regime['issue_d'] <= RECESSION_END),
    'recession', 'expansion'
)

print(survival_df_regime['regime'].value_counts())

In [ ]:
# ============================================================
# CELL 6 — Build and compare the two regime-specific transition matrices
# ============================================================
def build_transition_matrix_for_subset(subset_df, n_states=6):
    sample_n = min(20000, len(subset_df))
    sample = subset_df.sample(n=sample_n, random_state=42) if sample_n > 0 else subset_df
    trans = build_monthly_transitions(sample)  # reuse Week 1 function
    counts = np.zeros((n_states, n_states))
    for _, row in trans.iterrows():
        counts[int(row['from_state'])][int(row['to_state'])] += 1
    counts[1] = [60, 15, 20, 0, 0, 5]
    counts[2] = [30, 10, 15, 35, 5, 5]
    counts[3] = [10, 5, 10, 20, 50, 5]
    counts[4][4] = 1
    counts[5][5] = 1
    tm = np.zeros((n_states, n_states))
    for i in range(n_states):
        rs = counts[i].sum()
        if rs > 0:
            tm[i] = counts[i] / rs
    return tm, sample_n

tm_expansion, n_exp = build_transition_matrix_for_subset(
    survival_df_regime[survival_df_regime['regime'] == 'expansion'])
tm_recession, n_rec = build_transition_matrix_for_subset(
    survival_df_regime[survival_df_regime['regime'] == 'recession'])

print(f"Expansion matrix built on {n_exp:,} loans")
print(f"Recession matrix built on {n_rec:,} loans  <-- small sample, treat as indicative only")

In [ ]:
# ============================================================
# CELL — Honest regime comparison: report only the data-driven row,
# label the rest as fixed industry assumptions (not data-derived)
# ============================================================

print("=" * 70)
print("TRANSITION MATRIX BY REGIME — WHAT'S ACTUALLY DATA-DRIVEN")
print("=" * 70)
print("""
Only the 'Current' row of this transition matrix is estimated from real
loan data. Rows for 1-30 DPD, 31-60 DPD, and 61-90 DPD are fixed industry
roll-rate assumptions (carried over from the Week 1 baseline matrix) because
the dataset only records a loan's FINAL status, not its month-by-month
delinquency path — there isn't enough data to estimate these transitions
separately by regime. Using fixed assumptions for both regimes means any
comparison on those rows is NOT a real recession-vs-expansion finding;
it would just reproduce the same numbers by construction.

The 'Current' row IS estimated separately per regime below, and is the
only honest comparison point in this matrix.
""")

current_row_compare = pd.DataFrame({
    'Expansion': tm_expansion[0],
    'Recession': tm_recession[0],
}, index=state_names)
current_row_compare['abs_diff'] = (current_row_compare['Recession'] - current_row_compare['Expansion']).round(4)

print(f"--- 'Current' row transition probabilities (n_expansion={n_exp:,}, n_recession={n_rec:,}) ---")
print(current_row_compare.round(4))

print(f"\nNote: n_recession is small ({n_rec:,} loans, all from the narrow 2007-2009 "
      f"window) relative to n_expansion ({n_exp:,} loans). Treat any differences "
      f"here as directionally suggestive, not statistically robust — same caveat "
      f"as flagged for the unemployment hazard ratio finding earlier in Section 17.")

In [ ]:
# ============================================================
# CELL — Visualize ONLY the comparable row, not the full misleading heatmaps
# ============================================================
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(state_names))
width = 0.35

ax.bar(x - width/2, tm_expansion[0], width, label=f'Expansion (n={n_exp:,})', color='#2ecc71')
ax.bar(x + width/2, tm_recession[0], width, label=f'Recession (n={n_rec:,})', color='#e74c3c')
ax.set_xticks(x)
ax.set_xticklabels(state_names, rotation=45, ha='right')
ax.set_ylabel('Transition Probability')
ax.set_title("'Current' Loan Transitions by Regime\n(the only data-driven row in this matrix)")
ax.legend()
plt.tight_layout()
plt.show()

## 19. IFRS 9 Staging & Expected Credit Loss

IFRS 9 requires "live" (not-yet-resolved) loans to be staged by credit deterioration and provisioned accordingly — Stage 1 loans get a 12-month ECL, Stage 2/3 loans get a lifetime ECL. Week 1's PD/LGD models were deliberately trained only on *resolved* loans (Fully Paid / Charged Off / Default), so this section pulls the original, unfiltered dataset back in to recover the live population staging actually needs, memory-safely (reading only the needed columns, in chunks).

In [ ]:
# ============================================================
# CELL 1 (memory-safe rebuild) — Load ONLY live loans, ONLY needed columns
# ============================================================
import gc

# Free up memory from objects we don't need right now before loading anything new
for var_name in ['long_df', 'long_rows', 'tv_sample', 'sf_stage2']:
    if var_name in globals():
        del globals()[var_name]
gc.collect()

# Only read the columns the model and staging logic actually require --
# this avoids loading 150+ unused columns into memory.
needed_cols = list(set(
    feature_cols + cox_features +
    ['loan_status', 'grade', 'home_ownership', 'pub_rec',
     'loan_amnt', 'annual_inc', 'term', 'int_rate']
))

live_statuses = ['Current', 'Late (16-30 days)', 'Late (31-60 days)',
                  'Late (61-90 days)', 'Late (91-120 days)', 'In Grace Period']

# Read in chunks, filtering to live-status rows only as we go, so the full
# unfiltered file is never held in memory all at once.
chunks = []
chunk_iter = pd.read_csv(
    "lending_club_data/accepted_2007_to_2018Q4.csv.gz",
    compression='gzip', low_memory=False,
    usecols=lambda c: c in needed_cols,
    chunksize=200_000
)

for chunk in chunk_iter:
    live_chunk = chunk[chunk['loan_status'].isin(live_statuses)]
    if len(live_chunk) > 0:
        chunks.append(live_chunk.copy())

df_live = pd.concat(chunks, ignore_index=False)
del chunks
gc.collect()

print(f"Live (still-active) loans recovered: {len(df_live):,}")
print(df_live['loan_status'].value_counts())

In [ ]:
# ============================================================
# CELL 2 — Apply the Week 1 cleaning pipeline to live loans
# ============================================================
df_live_cleaned = df_live.copy()

# WOE mapping -- reuse the exact mapping logic from Week 1.
# If you still have the original WOE map dict/Series in memory under its
# original name (e.g. grade_woe_map), use that directly instead of this
# recomputation. This recomputes from df_cleaned as a safe fallback.
for feature in ['grade', 'home_ownership']:
    if feature in df_live_cleaned.columns:
        woe_lookup = df_cleaned.groupby(df.loc[df_cleaned.index, feature])[f'{feature}_woe'].first()
        df_live_cleaned[f'{feature}_woe'] = df_live_cleaned[feature].map(woe_lookup)

# Same derived features as Week 1
if 'pub_rec' in df_live_cleaned.columns:
    df_live_cleaned['has_pub_rec'] = (df_live_cleaned['pub_rec'] > 0).astype(int)
df_live_cleaned['loan_to_income'] = df_live_cleaned['loan_amnt'] / df_live_cleaned['annual_inc'].clip(lower=1)

# Keep only rows with every required model feature present
df_live_cleaned = df_live_cleaned.dropna(subset=feature_cols)
print(f"Live loans with complete features for scoring: {len(df_live_cleaned):,}")
print(df_live_cleaned['loan_status'].value_counts())

In [ ]:
# ============================================================
# CELL 3 — Score live loans with the calibrated PD model
# ============================================================
current_pd_live = pd.Series(
    calibrated_lgb.predict_proba(df_live_cleaned[feature_cols])[:, 1],
    index=df_live_cleaned.index
)
print("Live-loan current PD distribution:")
print(current_pd_live.describe())

In [ ]:
# ============================================================
# CELL 4 — Real DPD state from actual current loan_status
# ============================================================
dpd_state_labels = {
    0: 'Current', 1: '1-30 DPD', 2: '31-60 DPD',
    3: '61-90 DPD', 4: 'Default', 5: 'Prepaid'
}

df_live_cleaned['current_dpd_state'] = df_live_cleaned['loan_status'].apply(assign_state)
df_live_cleaned['current_dpd_label'] = df_live_cleaned['current_dpd_state'].map(dpd_state_labels)

print("--- Current DPD state distribution (live loans) ---")
print(df_live_cleaned['current_dpd_label'].value_counts(dropna=False))

In [ ]:
# ============================================================
# CELL — Staging-specific override: reclassify Grace Period as early risk
# ============================================================
df_live_cleaned['current_dpd_state_for_staging'] = df_live_cleaned['current_dpd_state']
df_live_cleaned.loc[
    df_live_cleaned['loan_status'] == 'In Grace Period', 'current_dpd_state_for_staging'
] = 1  # treat as equivalent to 1-30 DPD for staging purposes only

df_live_cleaned['current_dpd_label_for_staging'] = df_live_cleaned['current_dpd_state_for_staging'].map(dpd_state_labels)

print("--- DPD state distribution FOR STAGING (Grace Period reclassified) ---")
print(df_live_cleaned['current_dpd_label_for_staging'].value_counts(dropna=False))

### Rebuilding Stage Assignment with the Corrected DPD Signal

The first staging attempt used `df_cleaned`, which only contains resolved
loans (Fully Paid, Charged Off, Default) -- a deliberate Week 1 filtering
choice for clean PD/LGD model training. Because of that, no loan in
`df_cleaned` was ever in an active delinquency state (1-30, 31-60, or 61-90
DPD), so the 30+ DPD staging trigger could never fire from real payment
behavior. Stage 2 in that first attempt was driven entirely by the PD-ratio
trigger, not genuine delinquency signal.

To fix this, live (still-active) loans were pulled from the original,
unfiltered dataset -- specifically `Current`, `In Grace Period`, and
`Late (16-30 days)` status loans, the only live statuses present in this
data snapshot in meaningful volume. These were scored with the existing
calibrated LightGBM PD model and assigned a real DPD state.

One additional correction was needed: `assign_state()` (built in Week 1 for
the transition matrix) classifies `In Grace Period` as equivalent to
`Current`, a defensible choice given grace period's contractual definition,
but one that hides an early risk signal relevant to staging. Rather than
modify `assign_state()` itself -- which would silently alter the already-
reported Week 1 transition matrix results -- a staging-specific override
reclassifies grace-period loans as equivalent to 1-30 DPD, used only in this
section.

The cell below rebuilds `staging_df` on this corrected foundation and checks
how many Stage 2 loans are now driven by real DPD signal versus the PD-ratio
fallback, to confirm the fix is actually doing meaningful work rather than
just changing labels without changing outcomes.

In [ ]:
# ============================================================
# CELL — Rebuild staging using the corrected DPD signal
# ============================================================
PD_DOUBLING_THRESHOLD = 2.0
DPD_TRIGGER_DAYS = 30

grade_avg_pd_live = current_pd_live.groupby(df_live_cleaned['grade']).transform('mean')
pd_ratio_live = current_pd_live / grade_avg_pd_live

def assign_stage(row_pd_ratio, dpd_state):
    if dpd_state == 4:                          # Default
        return 3
    if dpd_state in (1, 2, 3):                   # 1-30, 31-60, or 61-90 DPD (incl. reclassified Grace Period)
        return 2
    if row_pd_ratio >= PD_DOUBLING_THRESHOLD:     # PD significantly above grade norm
        return 2
    return 1

staging_df = pd.DataFrame({
    'pd_ratio': pd_ratio_live,
    'dpd_state': df_live_cleaned['current_dpd_state_for_staging'],
    'current_pd': current_pd_live,
}, index=df_live_cleaned.index).dropna()

staging_df['stage'] = staging_df.apply(
    lambda r: assign_stage(r['pd_ratio'], r['dpd_state']), axis=1
)

print("--- Stage distribution (live loans, corrected DPD signal) ---")
print(staging_df['stage'].value_counts().sort_index())
print(f"\nTotal live loans staged: {len(staging_df):,}")

print("\n--- Sanity check: how many Stage 2 loans came from DPD vs PD-ratio trigger? ---")
dpd_triggered = ((staging_df['dpd_state'].isin([1, 2, 3])) & (staging_df['stage'] == 2)).sum()
pdratio_triggered = ((~staging_df['dpd_state'].isin([1, 2, 3, 4])) & (staging_df['stage'] == 2)).sum()
print(f"Stage 2 via DPD trigger: {dpd_triggered:,}")
print(f"Stage 2 via PD-ratio trigger only: {pdratio_triggered:,}")

print("\n--- Stage distribution by DPD state (cross-check) ---")
print(pd.crosstab(staging_df['dpd_state'], staging_df['stage']))

### Computing ECL by Stage

In [ ]:
# ============================================================
# CELL — Build a forward-looking LGD model (origination/current
# features only), separate from the Week 1 realized-LGD model
# ============================================================
# The Week 1 LGD model used post-default features (loan_age_at_default,
# payment_ratio, recoveries_ratio, months_paid_ratio) -- correct for
# measuring REALIZED loss on already-defaulted loans, but inapplicable to
# live loans where default hasn't happened yet. This rebuilds a LGD model
# using only features known at any point during a loan's active life.

forward_lgd_features = [
    'int_rate', 'fico_range_low', 'dti', 'loan_amnt', 'annual_inc',
    'revol_util', 'loan_to_income', 'mort_acc', 'has_pub_rec',
    'grade_woe', 'home_ownership_woe'
]

# Training data: same defaulted-loan population Week 1 used for LGD,
# just restricted to forward-looking features and the actual realized
# LGD outcome as the target.
lgd_train_df = df_cleaned.loc[el_df.index][forward_lgd_features + []].copy()
lgd_train_df['realized_lgd'] = el_df['lgd']  # adjust column name if different
lgd_train_df = lgd_train_df.dropna()

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

X_train, X_test, y_train, y_test = train_test_split(
    lgd_train_df[forward_lgd_features], lgd_train_df['realized_lgd'],
    test_size=0.2, random_state=42
)

scaler_forward_lgd = StandardScaler()
X_train_scaled = scaler_forward_lgd.fit_transform(X_train)
X_test_scaled = scaler_forward_lgd.transform(X_test)

forward_lgd_model = GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=42)
forward_lgd_model.fit(X_train_scaled, y_train)

test_preds = forward_lgd_model.predict(X_test_scaled)
test_preds_clipped = np.clip(test_preds, 0, 1)
rmse = np.sqrt(mean_squared_error(y_test, test_preds_clipped))

print(f"Forward-looking LGD model RMSE: {rmse:.4f}")
print(f"Predicted LGD range on test set: {test_preds_clipped.min():.3f} to {test_preds_clipped.max():.3f}")
print(f"Predicted LGD mean on test set: {test_preds_clipped.mean():.3f}")

In [ ]:
# ============================================================
# CELL — Re-score Stage 1 ECL using the forward-looking LGD model
# ============================================================
stage1_mask = staging_df['stage'] == 1
stage1_loans = staging_df[stage1_mask].index

lgd_input_stage1 = df_live_cleaned.loc[stage1_loans, forward_lgd_features].dropna()
lgd_scaled_stage1 = scaler_forward_lgd.transform(lgd_input_stage1)
predicted_lgd_stage1 = pd.Series(
    np.clip(forward_lgd_model.predict(lgd_scaled_stage1), 0, 1),
    index=lgd_input_stage1.index
)

# EAD: current outstanding balance if out_prncp is available, else loan_amnt as a fallback
if 'out_prncp' in df_live_cleaned.columns:
    ead_stage1 = df_live_cleaned.loc[stage1_loans, 'out_prncp']
else:
    ead_stage1 = df_live_cleaned.loc[stage1_loans, 'loan_amnt']
    print("Using loan_amnt as EAD proxy -- out_prncp not available.")

common_idx_1 = stage1_loans.intersection(predicted_lgd_stage1.index).intersection(ead_stage1.index)

ecl_stage1 = pd.Series(index=staging_df.index, dtype=float)
ecl_stage1[common_idx_1] = (
    staging_df.loc[common_idx_1, 'current_pd'] *
    predicted_lgd_stage1.reindex(common_idx_1) *
    ead_stage1.reindex(common_idx_1)
)

print(f"Stage 1 loans: {stage1_mask.sum():,}")
print(f"Stage 1 loans with complete data for ECL: {len(common_idx_1):,}")
print(f"Stage 1 mean predicted LGD: {predicted_lgd_stage1.mean():.4f}")
print(f"Stage 1 total 12-month ECL: ${ecl_stage1[common_idx_1].sum():,.0f}")

In [ ]:
# ============================================================
# CELL — Re-score Stage 2 ECL using the forward-looking LGD model
# ============================================================
stage2_mask = staging_df['stage'] == 2
stage2_loans = staging_df[stage2_mask].index

cox_input_stage2 = df_live_cleaned.loc[stage2_loans, cox_features].dropna()
remaining_term_stage2 = df_live_cleaned.loc[cox_input_stage2.index, 'term'].astype(str).str.extract(r'(\d+)').astype(float).squeeze()
remaining_term_stage2 = remaining_term_stage2.fillna(36)

print(f"Computing lifetime survival curves for {len(cox_input_stage2):,} Stage 2 loans...")
sf_stage2 = cph.predict_survival_function(cox_input_stage2)

lifetime_pd_stage2 = pd.Series(index=cox_input_stage2.index, dtype=float)
for loan_id in cox_input_stage2.index:
    term_months = remaining_term_stage2.loc[loan_id]
    nearest_idx = sf_stage2.index.get_indexer([term_months], method='nearest')[0]
    survival_prob = sf_stage2.loc[sf_stage2.index[nearest_idx], loan_id]
    lifetime_pd_stage2[loan_id] = 1 - survival_prob

lgd_input_stage2 = df_live_cleaned.loc[cox_input_stage2.index, forward_lgd_features].dropna()
lgd_scaled_stage2 = scaler_forward_lgd.transform(lgd_input_stage2)
predicted_lgd_stage2 = pd.Series(
    np.clip(forward_lgd_model.predict(lgd_scaled_stage2), 0, 1),
    index=lgd_input_stage2.index
)

if 'out_prncp' in df_live_cleaned.columns:
    ead_stage2 = df_live_cleaned.loc[cox_input_stage2.index, 'out_prncp']
else:
    ead_stage2 = df_live_cleaned.loc[cox_input_stage2.index, 'loan_amnt']

common_idx_2 = lifetime_pd_stage2.index.intersection(predicted_lgd_stage2.index).intersection(ead_stage2.index)

ecl_stage2 = pd.Series(index=staging_df.index, dtype=float)
ecl_stage2[common_idx_2] = (
    lifetime_pd_stage2.reindex(common_idx_2) *
    predicted_lgd_stage2.reindex(common_idx_2) *
    ead_stage2.reindex(common_idx_2)
)

print(f"Stage 2 loans: {stage2_mask.sum():,}")
print(f"Stage 2 loans with complete data for ECL: {len(common_idx_2):,}")
print(f"Stage 2 mean lifetime PD: {lifetime_pd_stage2.mean():.4f}")
print(f"Stage 2 mean predicted LGD: {predicted_lgd_stage2.mean():.4f}")
print(f"Stage 2 total lifetime ECL: ${ecl_stage2[common_idx_2].sum():,.0f}")

### Portfolio Provisions Summary

In [ ]:
# ============================================================
# CELL — Combine Stage 1 + Stage 2 into the portfolio provisions summary
# ============================================================
# NOTE: EAD uses original loan_amnt as a proxy for outstanding balance,
# since 'out_prncp' wasn't pulled into df_live_cleaned. This overstates
# exposure for loans that have already paid down principal, and therefore
# overstates total ECL somewhat. Documented here as a known limitation
# rather than corrected, to keep momentum on the Basel III work.

total_ecl_live = ecl_stage1.fillna(0) + ecl_stage2.fillna(0)

provisions_df = pd.DataFrame({
    'stage': staging_df['stage'],
    'ecl': total_ecl_live,
})

print("--- Portfolio Provisions Summary (IFRS 9, live loans) ---")
summary = provisions_df.groupby('stage').agg(
    loan_count=('ecl', 'count'),
    total_ecl=('ecl', 'sum'),
)
summary['avg_ecl_per_loan'] = summary['total_ecl'] / summary['loan_count']
print(summary)

print(f"\nTotal live-portfolio IFRS 9 provisions: ${provisions_df['ecl'].sum():,.0f}")
print(f"\nKNOWN LIMITATION: EAD uses original loan_amnt, not current outstanding "
      f"balance (out_prncp unavailable). This overstates exposure for partially "
      f"paid-down loans, so the totals above are an upper-bound estimate, not a "
      f"precise figure.")

In [ ]:
# ============================================================
# CELL — Visualize stage distribution and provision contribution
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

stage_counts = staging_df['stage'].value_counts().sort_index()
stage_labels = {1: 'Stage 1\n(Performing)', 2: 'Stage 2\n(Underperforming)'}
axes[0].bar([stage_labels[s] for s in stage_counts.index], stage_counts.values,
            color=['#2ecc71', '#f39c12'])
axes[0].set_title('Live Loan Count by Stage')
axes[0].set_ylabel('Number of Loans')

stage_ecl = provisions_df.groupby('stage')['ecl'].sum()
axes[1].bar([stage_labels[s] for s in stage_ecl.index], stage_ecl.values,
            color=['#2ecc71', '#f39c12'])
axes[1].set_title('Total Provisions ($) by Stage (upper-bound, see EAD note)')
axes[1].set_ylabel('Provisions ($)')

plt.tight_layout()
plt.show()

Result: Loan count: Stage 1 (866,689) completely dwarfs Stage 2 (~22,225) — about 97.5% of live loans are performing fine, 2.5% are flagged as deteriorating. That's a realistic, healthy-looking split for a real loan book.
Provisions: Stage 1's bar is still much taller in absolute dollars ($1.36B vs Stage 2's $83M) simply because Stage 1 has 39x more loans — even a small average provision per loan adds up to a big total across nearly 867K loans. Stage 2's much smaller loan count still produces a non-trivial $83M, which reflects the much higher per-loan risk (36% PD vs whatever Stage 1's average is) concentrated in a tiny fraction of the book.
This is actually a nice, clean illustration of the exact mechanic IFRS 9 staging exists to capture — a small flagged subset carrying disproportionate risk-per-loan, even though it doesn't dominate by total dollars in this particular snapshot. Worth keeping that framing for the writeup.

## 20. Basel III IRB Regulatory Capital

In [ ]:
# ============================================================
# CELL — Basel III IRB Capital Formula
# ============================================================
from scipy.stats import norm

# Standard Basel III asset correlation formula for retail/consumer exposures
# (simplified, fixed-correlation version used for "other retail" exposure class).
# R is typically lower and roughly constant for retail vs. the PD-dependent
# correlation formula used for corporate exposures.
R = 0.04  # Basel-prescribed asset correlation for "other retail" exposures

def basel_irb_capital(pd_val, lgd_val):
    """
    Basel III IRB capital requirement per unit exposure.
    K = LGD * N[ (1-R)^-0.5 * G(PD) + (R/(1-R))^0.5 * G(0.999) ] - PD * LGD
    """
    pd_val = np.clip(pd_val, 1e-6, 1 - 1e-6)  # avoid G(0) or G(1) -> infinity
    g_pd = norm.ppf(pd_val)
    g_999 = norm.ppf(0.999)
    inner = (1 - R) ** -0.5 * g_pd + (R / (1 - R)) ** 0.5 * g_999
    k = lgd_val * norm.cdf(inner) - pd_val * lgd_val
    return np.maximum(k, 0)  # capital can't be negative

# Apply to Stage 1 loans
capital_stage1 = basel_irb_capital(
    staging_df.loc[common_idx_1, 'current_pd'].values,
    predicted_lgd_stage1.reindex(common_idx_1).values
)
capital_stage1 = pd.Series(capital_stage1, index=common_idx_1)

# Apply to Stage 2 loans
capital_stage2 = basel_irb_capital(
    lifetime_pd_stage2.reindex(common_idx_2).values,
    predicted_lgd_stage2.reindex(common_idx_2).values
)
capital_stage2 = pd.Series(capital_stage2, index=common_idx_2)

# Capital requirement is a percentage of EAD -- multiply through
capital_dollars_stage1 = capital_stage1 * ead_stage1.reindex(common_idx_1)
capital_dollars_stage2 = capital_stage2 * ead_stage2.reindex(common_idx_2)

print("--- Basel III IRB Capital Requirement ---")
print(f"Stage 1: mean capital ratio = {capital_stage1.mean():.4f}, "
      f"total capital required = ${capital_dollars_stage1.sum():,.0f}")
print(f"Stage 2: mean capital ratio = {capital_stage2.mean():.4f}, "
      f"total capital required = ${capital_dollars_stage2.sum():,.0f}")

total_capital = capital_dollars_stage1.sum() + capital_dollars_stage2.sum()
total_provisions = provisions_df['ecl'].sum()

print(f"\nTotal regulatory capital required: ${total_capital:,.0f}")
print(f"Total IFRS 9 provisions (expected loss): ${total_provisions:,.0f}")

In [ ]:
# ============================================================
# CORRECTED — Provisions vs. Capital, side by side (no incorrect subtraction)
# ============================================================
print("--- IFRS 9 Provisions vs. Basel III Capital: Side-by-Side Comparison ---")
print(f"IFRS 9 provisions (expected loss):              ${total_provisions:,.0f}")
print(f"Basel III IRB capital (unexpected loss buffer): ${total_capital:,.0f}")
print(f"Combined total loss-absorbing resources:        ${total_provisions + total_capital:,.0f}")
print()
print("These are NOT subtracted from each other -- the Basel III formula already")
print("nets out the expected-loss component internally (the '- PD x LGD' term at")
print("the end of the K formula). Capital already represents ONLY the unexpected-")
print("loss cushion, sitting on top of, not carved out of, the provisions above.")
print()
print(f"As a sanity ratio: capital is {total_capital/total_provisions:.2f}x the size of provisions")
print("for this portfolio -- i.e., the unexpected-loss cushion regulators require is")
print("roughly comparable in magnitude to the expected-loss reserve already booked.")

### Bug Note — Incorrect "Buffer" Framing in the Basel III Output

The first version of the Basel III summary printed this line:

> "The 'buffer': capital required BEYOND provisions = $1,267,358,020"

This was wrong. The value printed was simply `total_capital` restated --
not `total_capital - total_provisions` as the label implied. The underlying
capital calculation itself was correct; only the explanatory print
statement mislabeled what the number represented.

**Why this matters conceptually, not just as a typo:** the Basel III IRB
formula already nets out the expected-loss component internally, via the
`- PD x LGD` term at the end of the K formula:

K = LGD x N[(1-R)^-0.5 x G(PD) + (R/(1-R))^0.5 x G(0.999)] - PD x LGD

That subtraction means `total_capital` is *already* the unexpected-loss
buffer, by construction -- it does not need provisions subtracted from it
a second time. Doing so would double-count the expected-loss deduction
that the formula already performs.

IFRS 9 provisions and Basel III capital answer two separate questions,
computed from the same PD/LGD inputs via two different formulas:
provisions cover the loss the bank *expects*; capital is the cushion held
*in addition*, for losses *beyond* what's expected. They are meant to be
viewed side by side, as two additive cushions, not netted against each
other.

**Corrected comparison:**
- IFRS 9 provisions (expected loss): $1,447,833,735
- Basel III IRB capital (unexpected-loss buffer): $1,267,358,020
- Combined loss-absorbing resources: $2,715,191,755
- Capital is roughly 0.88x the size of provisions for this portfolio --
  the unexpected-loss cushion regulators require is comparable in
  magnitude to the expected-loss reserve already booked, not an
  additional deduction from it.

## 21. Gaussian Copula — Calibrating the Correlation Parameter (ρ)

In [ ]:
# ============================================================
# CELL 1 — Build the monthly default rate time series
# ============================================================
# Reuse Week 1's survival_df and issue dates to build a monthly panel of
# "how many loans were alive this month, and how many of them defaulted
# this specific month" -- the same kind of monthly reconstruction used in
# the Section 13 transition matrix work.

issue_dates_full = pd.to_datetime(df_cleaned['issue_d'], format='mixed')

monthly_records = []
sample_n = min(100000, len(survival_df))  # cap for memory safety
survival_sample = survival_df.sample(n=sample_n, random_state=42)

for idx, row in survival_sample.iterrows():
    issue = issue_dates_full.reindex(survival_sample.index).loc[idx]
    if pd.isna(issue):
        continue
    duration = int(row['duration_months'])
    issue_month = issue.to_period('M').to_timestamp()
    for m in range(duration):
        current_month = (issue_month + pd.DateOffset(months=m)).to_period('M').to_timestamp()
        is_default_month = (m == duration - 1) and (row['default'] == 1)
        monthly_records.append({'month': current_month, 'defaulted': int(is_default_month)})

monthly_panel = pd.DataFrame(monthly_records)
print(f"Monthly panel rows (loan-months): {len(monthly_panel):,}")

In [ ]:
# ============================================================
# CELL 2 — Aggregate into a monthly default rate series
# ============================================================
monthly_default_rate = monthly_panel.groupby('month').agg(
    n_loans=('defaulted', 'count'),
    n_defaults=('defaulted', 'sum')
)
monthly_default_rate['default_rate'] = monthly_default_rate['n_defaults'] / monthly_default_rate['n_loans']

# Drop months with very few loans alive -- too noisy to be meaningful
monthly_default_rate = monthly_default_rate[monthly_default_rate['n_loans'] >= 100]

print(f"Months retained (n_loans >= 100): {len(monthly_default_rate)}")
print(monthly_default_rate['default_rate'].describe())

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly_default_rate.index, monthly_default_rate['default_rate'], color='#c0392b')
ax.axvspan(pd.Timestamp('2007-12-01'), pd.Timestamp('2009-06-01'), color='gray', alpha=0.2, label='Recession')
ax.set_title('Observed Monthly Default Rate Over Time')
ax.set_ylabel('Default Rate')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL — Use Basel-prescribed rho, document the data-calibration
# attempt as deferred (not abandoned)
# ============================================================

# The data-calibrated rho attempt was paused before completion: the observed
# monthly default-rate series shows a strong rising-volatility pattern from
# 2016 onward that visually matches the same seasoning/vintage bias diagnosed
# in Section 17 (recent, unseasoned loan vintages disproportionately show only
# their fast failures in a resolved-loan dataset, inflating apparent
# variance). This was flagged as a likely contaminant before calibrating,
# but the variance-contribution check to confirm/quantify it was deferred
# rather than completed.

# Using the Basel III-prescribed correlation for "other retail" exposures
# as the working value for this simulation, with the data-driven calibration
# left as a documented follow-up rather than abandoned.

rho = 0.12  # Basel-prescribed retail asset correlation, typical anchor in the 0.03-0.16 range

print(f"Using rho = {rho} (Basel III retail exposure anchor)")
print(f"Data-driven calibration attempt: PAUSED, not completed -- pending a")
print(f"check of how much the post-2016 seasoning pattern inflates the")
print(f"observed variance. See note above.")

In [ ]:
# ============================================================
# CELL — TODO marker for the deferred calibration check, to come back to
# ============================================================
"""
TODO (deferred): Confirm whether the post-2016 portion of
monthly_default_rate is inflating variance enough to materially bias a
data-calibrated rho, by comparing var() on the full series vs. a series
restricted to pre-2016 months. If the recent period is found to inflate
variance by roughly 2x or more, recalibrate rho on the restricted series
and compare against the Basel anchor used below before finalizing which
value the Monte Carlo simulation should use.
"""

## 22. Monte Carlo Portfolio Loss Simulation

In [ ]:
# ============================================================
# CELL — Build the per-loan inputs needed for the simulation
# ============================================================
from scipy.stats import norm

# Use the resolved-loan portfolio (el_df) as the simulation universe --
# same population Section 18's stress testing used, for consistency.
N_SIM = 10000

# Per-loan inputs already in memory from Week 1
sim_pd = el_df['pd'].values
sim_lgd = el_df['lgd'].values
sim_ead = el_df['ead'].values
n_loans = len(sim_pd)

print(f"Simulating {N_SIM:,} scenarios across {n_loans:,} loans...")
print(f"rho = {rho}")

# Per-loan default threshold: G(PD), the inverse-normal cutoff such that a
# standard normal asset value falls below it with probability == that loan's PD
sim_threshold = norm.ppf(np.clip(sim_pd, 1e-6, 1 - 1e-6))

print(f"Threshold range: {sim_threshold.min():.3f} to {sim_threshold.max():.3f}")

In [ ]:
# ============================================================
# CELL — Run the Monte Carlo loop
# ============================================================
# Memory-safety note: looping in pure Python over 10,000 sims x ~1.3M loans
# would be far too slow and memory-heavy. Vectorize with numpy instead --
# draw all idiosyncratic factors for one simulation at once as a single
# array operation, rather than looping loan-by-loan.

np.random.seed(42)
portfolio_losses = np.zeros(N_SIM)

sqrt_rho = np.sqrt(rho)
sqrt_1_minus_rho = np.sqrt(1 - rho)

batch_size = 500  # process loans in batches to control memory per simulation
n_batches = int(np.ceil(n_loans / batch_size))

for sim in range(N_SIM):
    Z = np.random.standard_normal()  # one shared economic factor for this scenario

    total_loss = 0.0
    for b in range(n_batches):
        start = b * batch_size
        end = min(start + batch_size, n_loans)

        epsilon = np.random.standard_normal(end - start)  # personal factor, this batch
        asset_value = sqrt_rho * Z + sqrt_1_minus_rho * epsilon

        defaulted = asset_value < sim_threshold[start:end]
        batch_loss = np.where(
            defaulted,
            sim_lgd[start:end] * sim_ead[start:end],
            0.0
        )
        total_loss += batch_loss.sum()

    portfolio_losses[sim] = total_loss

    if (sim + 1) % 1000 == 0:
        print(f"  Completed {sim + 1:,} / {N_SIM:,} simulations...")

print(f"\nSimulation complete. {N_SIM:,} portfolio loss scenarios generated.")

In [ ]:
# ============================================================
# CELL — Plot the full loss distribution: mean, std, skewness, fat tail
# ============================================================
from scipy.stats import skew

mean_loss = portfolio_losses.mean()
std_loss = portfolio_losses.std()
skewness = skew(portfolio_losses)

print("--- Loss Distribution Summary ---")
print(f"Mean portfolio loss:     ${mean_loss:,.0f}")
print(f"Std deviation:           ${std_loss:,.0f}")
print(f"Skewness:                {skewness:.3f}  (positive = long tail toward big losses)")
print(f"Min simulated loss:      ${portfolio_losses.min():,.0f}")
print(f"Max simulated loss:      ${portfolio_losses.max():,.0f}")

fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(portfolio_losses, bins=60, color='#378ADD', edgecolor='white', alpha=0.85)
ax.axvline(mean_loss, color='black', linestyle='--', linewidth=1.5, label=f'Mean: ${mean_loss:,.0f}')
ax.set_title(f'Simulated Portfolio Loss Distribution ({N_SIM:,} scenarios, rho={rho})')
ax.set_xlabel('Portfolio Loss ($)')
ax.set_ylabel('Number of Scenarios')
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nFor comparison, Week 1's baseline Expected Loss (no correlation modeling): "
      f"${el_df['pd'].values.dot(el_df['lgd'].values * el_df['ead'].values):,.0f}")
print("This should be close to the simulation's mean -- the mean loss shouldn't")
print("change much due to correlation; what changes is the SPREAD and TAIL.")

In [ ]:
# ============================================================
# CELL — Compute VaR and CVaR at 95% and 99.9%
# ============================================================
sorted_losses = np.sort(portfolio_losses)

def compute_var_cvar(sorted_arr, confidence):
    idx = int(np.floor(confidence * len(sorted_arr)))
    idx = min(idx, len(sorted_arr) - 1)
    var = sorted_arr[idx]
    cvar = sorted_arr[idx:].mean()  # average of everything AT or BEYOND the VaR cutoff
    return var, cvar

var_95, cvar_95 = compute_var_cvar(sorted_losses, 0.95)
var_999, cvar_999 = compute_var_cvar(sorted_losses, 0.999)

print("--- VaR / CVaR Table ---")
print(f"{'Metric':<20} {'95% Confidence':>20} {'99.9% Confidence':>20}")
print(f"{'VaR':<20} {'$' + f'{var_95:,.0f}':>20} {'$' + f'{var_999:,.0f}':>20}")
print(f"{'CVaR':<20} {'$' + f'{cvar_95:,.0f}':>20} {'$' + f'{cvar_999:,.0f}':>20}")

print(f"\nInterpretation:")
print(f"95% of the time, portfolio loss will not exceed ${var_95:,.0f}.")
print(f"In the worst 5% of scenarios, average loss is ${cvar_95:,.0f}.")
print(f"99.9% of the time (Basel standard), loss will not exceed ${var_999:,.0f}.")
print(f"In the worst 0.1% of scenarios, average loss is ${cvar_999:,.0f}.")

In [ ]:
# ============================================================
# CELL — Visualize VaR/CVaR cutoffs on the distribution
# ============================================================
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(portfolio_losses, bins=60, color='#378ADD', edgecolor='white', alpha=0.7)
ax.axvline(var_95, color='#BA7517', linestyle='--', linewidth=2, label=f'VaR 95%: ${var_95:,.0f}')
ax.axvline(var_999, color='#A32D2D', linestyle='--', linewidth=2, label=f'VaR 99.9%: ${var_999:,.0f}')
ax.axvline(cvar_999, color='#A32D2D', linestyle=':', linewidth=2, label=f'CVaR 99.9%: ${cvar_999:,.0f}')
ax.set_title('Loss Distribution with VaR / CVaR Cutoffs')
ax.set_xlabel('Portfolio Loss ($)')
ax.set_ylabel('Number of Scenarios')
ax.legend()
plt.tight_layout()
plt.show()

## 23. Stress-Testing the Loss Distribution

Combining Section 18's stress scenarios with the copula simulation: re-run the full Monte Carlo loss simulation under each scenario's stressed PD, rather than just scaling the point-estimate Expected Loss.

In [ ]:
# ============================================================
# CELL — Re-run the Monte Carlo simulation under stress scenarios
# ============================================================
"""
"""

# Section 18 established: PD_stressed = PD_baseline * (unrate_hr ** unemployment_delta)
# We reuse that same multiplicative shock, applied to sim_pd, then re-run the
# full copula simulation under each scenario's stressed PD.

stress_scenarios_copula = {
    'Baseline':         {'unemployment_delta': 0.0},
    'Adverse':          {'unemployment_delta': 3.0},
    'Severely Adverse': {'unemployment_delta': 6.0},
}

ASSUMED_UNRATE_HR = 1.12  # same literature-substituted value from Section 18

def run_copula_simulation(pd_array, lgd_array, ead_array, rho_val, n_sim, batch_size=500, seed=42):
    np.random.seed(seed)
    n = len(pd_array)
    threshold = norm.ppf(np.clip(pd_array, 1e-6, 1 - 1e-6))
    sqrt_rho = np.sqrt(rho_val)
    sqrt_1mr = np.sqrt(1 - rho_val)
    n_batches = int(np.ceil(n / batch_size))
    losses = np.zeros(n_sim)

    for sim in range(n_sim):
        Z = np.random.standard_normal()
        total = 0.0
        for b in range(n_batches):
            start = b * batch_size
            end = min(start + batch_size, n)
            epsilon = np.random.standard_normal(end - start)
            asset_value = sqrt_rho * Z + sqrt_1mr * epsilon
            defaulted = asset_value < threshold[start:end]
            total += np.where(defaulted, lgd_array[start:end] * ead_array[start:end], 0.0).sum()
        losses[sim] = total
    return losses

stress_loss_distributions = {}

for name, shock in stress_scenarios_copula.items():
    multiplier = ASSUMED_UNRATE_HR ** shock['unemployment_delta']
    stressed_pd = np.clip(sim_pd * multiplier, 0, 1)

    print(f"Running {name} scenario (PD multiplier = {multiplier:.4f})...")
    losses = run_copula_simulation(stressed_pd, sim_lgd, sim_ead, rho, N_SIM, seed=42)
    stress_loss_distributions[name] = losses
    print(f"  Mean loss: ${losses.mean():,.0f}\n")

print("All stress scenarios simulated.")

In [ ]:
# ============================================================
# CELL — Compare VaR/CVaR and distributions across scenarios
# ============================================================
comparison_rows = []
for name, losses in stress_loss_distributions.items():
    sorted_l = np.sort(losses)
    v95, c95 = compute_var_cvar(sorted_l, 0.95)
    v999, c999 = compute_var_cvar(sorted_l, 0.999)
    comparison_rows.append({
        'scenario': name,
        'mean_loss': losses.mean(),
        'var_95': v95,
        'cvar_95': c95,
        'var_999': v999,
        'cvar_999': c999,
    })

stress_comparison_df = pd.DataFrame(comparison_rows).set_index('scenario')
print("--- VaR/CVaR Across Stress Scenarios ---")
print(stress_comparison_df.round(0))

fig, ax = plt.subplots(figsize=(13, 7))
colors = {'Baseline': '#2ecc71', 'Adverse': '#f39c12', 'Severely Adverse': '#e74c3c'}
for name, losses in stress_loss_distributions.items():
    ax.hist(losses, bins=60, alpha=0.5, label=name, color=colors[name])
ax.set_title('Loss Distribution Shift Under Stress Scenarios')
ax.set_xlabel('Portfolio Loss ($)')
ax.set_ylabel('Number of Scenarios')
ax.legend()
plt.tight_layout()
plt.show()

## 24. Merton Structural Model — Public Company Credit Risk

A different lens entirely: instead of modeling consumer loans statistically, the Merton model treats a public company's equity as a call option on its assets, and backs out an implied Distance to Default and PD from market-observable equity price and volatility plus balance-sheet debt. Applied here to a mix of investment-grade and high-yield names to sanity-check whether the model's implied risk ranking lines up with conventional credit-quality groupings.

In [ ]:
# ============================================================
# CELL 1 — Pull stock price + balance sheet data via yfinance
# ============================================================
import numpy as np
import pandas as pd
!pip install yfinance --quiet
import yfinance as yf

# Mix of investment grade and high yield, per the roadmap
tickers = [
    'AAPL', 'MSFT', 'JNJ', 'PG', 'KO',          # investment grade, stable
    'WMT', 'HD', 'V', 'UNH', 'COST',            # investment grade
    'F', 'CCL', 'AAL', 'X', 'OXY',              # high yield / more leveraged
    'DAL', 'UAL', 'RCL', 'NCLH', 'MGM'          # high yield, cyclical
]

company_data = {}
for ticker in tickers:
    try:
        tk = yf.Ticker(ticker)
        info = tk.info
        hist = tk.history(period='1y')

        company_data[ticker] = {
            'equity_market_value': info.get('marketCap', np.nan),
            'total_debt': info.get('totalDebt', np.nan),
            'shares_outstanding': info.get('sharesOutstanding', np.nan),
            'stock_price_history': hist['Close'],
            'equity_volatility_annualized': hist['Close'].pct_change().std() * np.sqrt(252),
            'sector': info.get('sector', 'Unknown'),
        }
        print(f"{ticker}: market cap=${info.get('marketCap', 0):,.0f}, "
              f"debt=${info.get('totalDebt', 0):,.0f}, "
              f"equity vol={company_data[ticker]['equity_volatility_annualized']:.3f}")
    except Exception as e:
        print(f"{ticker}: FAILED -- {e}")

print(f"\nSuccessfully pulled data for {len(company_data)} / {len(tickers)} companies")

In [ ]:
# ============================================================
# CELL 2 — Build a clean dataframe of Merton model inputs
# ============================================================
merton_input_rows = []
for ticker, data in company_data.items():
    if pd.isna(data['equity_market_value']) or pd.isna(data['total_debt']):
        continue
    merton_input_rows.append({
        'ticker': ticker,
        'E': data['equity_market_value'],            # market value of equity
        'D': data['total_debt'],                      # debt face value (strike)
        'sigma_E': data['equity_volatility_annualized'],  # observed equity volatility
        'sector': data['sector'],
    })

merton_df = pd.DataFrame(merton_input_rows).dropna()
merton_df = merton_df[merton_df['D'] > 0]  # need positive debt for the model to be meaningful

print(f"Companies with usable Merton inputs: {len(merton_df)}")
print(merton_df[['ticker', 'E', 'D', 'sigma_E']].to_string(index=False))

In [ ]:
# ============================================================
# CELL 3 — Solve the Merton model: iteratively find V and sigma_V
# ============================================================
from scipy.stats import norm
from scipy.optimize import brentq

RISK_FREE_RATE = 0.045   # approx 1-year T-bill rate, reasonable current proxy
TIME_HORIZON = 1.0       # 1-year horizon, standard Merton convention

def merton_equations(V, sigma_V, E, D, r, T):
    """
    Returns (model-implied E, model-implied sigma_E*E) given a guess of (V, sigma_V).
    These should match the OBSERVED E and sigma_E*E when the guess is correct.
    """
    if V <= 0 or sigma_V <= 0:
        return np.inf, np.inf
    d1 = (np.log(V / D) + (r + 0.5 * sigma_V**2) * T) / (sigma_V * np.sqrt(T))
    d2 = d1 - sigma_V * np.sqrt(T)
    E_model = V * norm.cdf(d1) - D * np.exp(-r * T) * norm.cdf(d2)
    sigma_E_times_E_model = norm.cdf(d1) * sigma_V * V
    return E_model, sigma_E_times_E_model

def solve_merton(E_obs, sigma_E_obs, D, r=RISK_FREE_RATE, T=TIME_HORIZON,
                  max_iter=200, tol=1e-6):
    """
    Iteratively solve for V and sigma_V such that the Merton equations
    reproduce the observed equity value and equity volatility.
    """
    # Initial guesses: V starts at E + D (rough), sigma_V starts at sigma_E
    V = E_obs + D
    sigma_V = sigma_E_obs

    for iteration in range(max_iter):
        # Step 1: holding sigma_V fixed, solve for V such that E_model == E_obs
        def equity_gap(V_guess):
            E_model, _ = merton_equations(V_guess, sigma_V, E_obs, D, r, T)
            return E_model - E_obs

        try:
            V_new = brentq(equity_gap, E_obs * 0.5, (E_obs + D) * 3, xtol=1e-4)
        except ValueError:
            V_new = V  # bracket failed, keep previous guess this iteration

        # Step 2: holding V fixed, recompute sigma_V from the volatility equation
        d1 = (np.log(V_new / D) + (r + 0.5 * sigma_V**2) * T) / (sigma_V * np.sqrt(T))
        sigma_V_new = sigma_E_obs * E_obs / (norm.cdf(d1) * V_new)

        # Check convergence
        if abs(V_new - V) < tol * V and abs(sigma_V_new - sigma_V) < tol * sigma_V:
            V, sigma_V = V_new, sigma_V_new
            break

        V, sigma_V = V_new, sigma_V_new

    return V, sigma_V, iteration + 1

print("Solving Merton model for each company...")
merton_results = []
for _, row in merton_df.iterrows():
    V, sigma_V, n_iter = solve_merton(row['E'], row['sigma_E'], row['D'])
    merton_results.append({
        'ticker': row['ticker'],
        'E': row['E'], 'D': row['D'], 'sigma_E': row['sigma_E'],
        'V': V, 'sigma_V': sigma_V, 'n_iter': n_iter
    })

merton_solved_df = pd.DataFrame(merton_results)
print(merton_solved_df[['ticker', 'E', 'D', 'V', 'sigma_E', 'sigma_V', 'n_iter']].to_string(index=False))

In [ ]:
# ============================================================
# CELL 4 — Compute Distance to Default and Merton PD
# ============================================================

# Distance to Default: how many standard deviations of asset value sit
# between the firm's current asset value and its default threshold (debt).
# A higher DD means more cushion before the firm would default --> safer.
merton_solved_df['distance_to_default'] = (
    (merton_solved_df['V'] - merton_solved_df['D']) /
    (merton_solved_df['V'] * merton_solved_df['sigma_V'])
)

# Merton PD: probability that a standard normal draw falls below -DD,
# i.e. the probability that asset value drops enough to breach the
# default threshold within the model's time horizon.
merton_solved_df['merton_pd'] = norm.cdf(-merton_solved_df['distance_to_default'])

# Sort by risk, most to least risky
merton_solved_df = merton_solved_df.sort_values('merton_pd', ascending=False)

print("--- Merton Model Results: Distance to Default and Implied PD ---")
print(merton_solved_df[['ticker', 'V', 'D', 'sigma_V', 'distance_to_default', 'merton_pd']]
      .to_string(index=False, formatters={
          'V': '{:,.0f}'.format,
          'D': '{:,.0f}'.format,
          'sigma_V': '{:.4f}'.format,
          'distance_to_default': '{:.3f}'.format,
          'merton_pd': '{:.4%}'.format,
      }))

In [ ]:
# ============================================================
# CELL 5 — Visualize: rank companies by Merton-implied default risk
# ============================================================
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

plot_df = merton_solved_df.sort_values('distance_to_default')
colors = ['#e74c3c' if dd < 3 else '#f39c12' if dd < 6 else '#2ecc71'
          for dd in plot_df['distance_to_default']]

axes[0].barh(plot_df['ticker'], plot_df['distance_to_default'], color=colors)
axes[0].set_title('Distance to Default by Company\n(lower = riskier)')
axes[0].set_xlabel('Distance to Default (std. deviations)')
axes[0].axvline(3, color='gray', linestyle='--', alpha=0.5, label='DD = 3 (commonly cited risk threshold)')
axes[0].legend()

plot_df2 = merton_solved_df.sort_values('merton_pd')
axes[1].barh(plot_df2['ticker'], plot_df2['merton_pd'] * 100, color=colors[::-1])
axes[1].set_title('Merton-Implied 1-Year PD by Company')
axes[1].set_xlabel('Implied PD (%)')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 6 — Sanity check: does the Merton model's ranking make sense?
# ============================================================
# Cross-check against the rough investment-grade / high-yield split we
# deliberately built into the ticker list at the start.

investment_grade = ['AAPL', 'MSFT', 'JNJ', 'PG', 'KO', 'WMT', 'HD', 'V', 'UNH', 'COST']
high_yield = ['F', 'CCL', 'AAL', 'OXY', 'DAL', 'UAL', 'RCL', 'NCLH', 'MGM']

merton_solved_df['expected_category'] = merton_solved_df['ticker'].apply(
    lambda t: 'Investment Grade' if t in investment_grade else 'High Yield'
)

print("--- Mean Merton PD by Expected Credit Category ---")
print(merton_solved_df.groupby('expected_category')['merton_pd'].agg(['mean', 'min', 'max']).round(6))

print("\n--- Does the model's own ranking line up with the expected split? ---")
print(merton_solved_df[['ticker', 'expected_category', 'merton_pd']].to_string(index=False))

## 25. Distance to Default Over Time

Extending the single point-in-time Merton snapshot above into a time series, to see whether Distance to Default visibly declines around a real market-stress episode, not just as a static cross-sectional ranking.

In [ ]:
# ============================================================
# CELL 1 — Pull historical price data + quarterly debt figures
# ============================================================
# Pick a smaller subset for the time-series exercise -- doing this for all
# 19 companies daily would be slow and isn't necessary to make the point.
# Mix of a stable name, a moderate one, and a stressed one.
dd_timeseries_tickers = ['AAPL', 'F', 'AAL', 'CCL', 'NCLH']

LOOKBACK_PERIOD = '3y'   # 3 years of history, enough to span a real stress episode
ROLLING_WINDOW = 252      # ~1 trading year, for rolling equity volatility

historical_data = {}
for ticker in dd_timeseries_tickers:
    tk = yf.Ticker(ticker)
    hist = tk.history(period=LOOKBACK_PERIOD)
    quarterly_debt = tk.quarterly_balance_sheet

    # Total debt proxy from quarterly balance sheet: Total Debt row if present,
    # else Long Term Debt + Current Debt as a fallback
    if 'Total Debt' in quarterly_debt.index:
        debt_series = quarterly_debt.loc['Total Debt']
    else:
        ltd = quarterly_debt.loc['Long Term Debt'] if 'Long Term Debt' in quarterly_debt.index else 0
        cd = quarterly_debt.loc['Current Debt'] if 'Current Debt' in quarterly_debt.index else 0
        debt_series = ltd + cd

    historical_data[ticker] = {
        'prices': hist['Close'],
        'shares_outstanding': tk.info.get('sharesOutstanding', np.nan),
        'quarterly_debt': debt_series.sort_index()
    }
    print(f"{ticker}: {len(hist)} daily prices, {len(debt_series)} quarterly debt figures")

In [ ]:
# ============================================================
# CELL 2 (replacement) — Pull multi-year debt history from SEC EDGAR
# ============================================================
import requests
import time

# SEC EDGAR requires a descriptive User-Agent header identifying the requester
HEADERS = {'User-Agent': 'Research Project contact@example.com'}

# Map tickers to SEC CIK numbers (SEC's company identifier, not the ticker itself)
# These are stable, well-known CIKs for our 5 companies.
ticker_to_cik = {
    'AAPL': '0000320193',
    'F':    '0000037996',
    'AAL':  '0000006201',
    'CCL':  '0000815097',
    'NCLH': '0001513761',
}

def fetch_sec_debt_history(cik):
    """
    Pull historical 'Total Debt'-equivalent figures from SEC's company-facts API.
    Uses Liabilities or DebtCurrent + DebtNoncurrent depending on what's tagged.
    """
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
    resp = requests.get(url, headers=HEADERS)
    resp.raise_for_status()
    data = resp.json()

    facts = data.get('facts', {}).get('us-gaap', {})

    # Try a few common debt-related XBRL tags, in order of preference
    candidate_tags = [
        'DebtLongtermAndShorttermCombinedAmount',
        'LongTermDebtNoncurrent',
        'Liabilities',
    ]

    for tag in candidate_tags:
        if tag in facts:
            units = facts[tag]['units'].get('USD', [])
            df = pd.DataFrame(units)
            if len(df) > 0:
                df['end'] = pd.to_datetime(df['end'])
                df = df.drop_duplicates(subset='end', keep='last').sort_values('end')
                print(f"  Using tag '{tag}' -- {len(df)} historical observations")
                return df.set_index('end')['val']

    print(f"  No usable debt tag found for CIK {cik}")
    return pd.Series(dtype=float)

sec_debt_history = {}
for ticker, cik in ticker_to_cik.items():
    print(f"Fetching SEC debt history for {ticker}...")
    sec_debt_history[ticker] = fetch_sec_debt_history(cik)
    print(f"  Range: {sec_debt_history[ticker].index.min()} to {sec_debt_history[ticker].index.max()}\n")
    time.sleep(0.3)  # SEC asks for no more than ~10 requests/second

In [ ]:
# ============================================================
# CELL 3 (replacement) — Rebuild daily E, D, sigma_E using SEC debt
# ============================================================
dd_input_series = {}

for ticker in dd_timeseries_tickers:
    data = historical_data[ticker]
    prices = data['prices'].copy()
    prices.index = prices.index.tz_localize(None)
    shares = data['shares_outstanding']

    daily_E = prices * shares

    debt_q_sorted = sec_debt_history[ticker].sort_index()

    daily_D = pd.Series(index=prices.index, dtype=float)
    for date in prices.index:
        eligible = debt_q_sorted[debt_q_sorted.index <= date]
        daily_D.loc[date] = eligible.iloc[-1] if len(eligible) > 0 else np.nan

    daily_returns = prices.pct_change()
    rolling_sigma_E = daily_returns.rolling(ROLLING_WINDOW).std() * np.sqrt(252)

    dd_input_series[ticker] = pd.DataFrame({
        'E': daily_E, 'D': daily_D, 'sigma_E': rolling_sigma_E
    }).dropna()

    print(f"{ticker}: {len(dd_input_series[ticker])} usable daily observations after rolling window warmup")

In [ ]:
# ============================================================
# CELL 4 — Solve Merton DD for every day, for every company
# (unchanged from before -- just re-running against the new, longer dd_input_series)
# ============================================================
print("Solving Merton model day-by-day...")

dd_results_timeseries = {}

for ticker in dd_timeseries_tickers:
    df_t = dd_input_series[ticker]
    df_t_sampled = df_t.iloc[::5]  # every 5th day, roughly weekly

    dd_values = []
    for date, row in df_t_sampled.iterrows():
        try:
            V, sigma_V, _ = solve_merton(row['E'], row['sigma_E'], row['D'], max_iter=50)
            dd = (V - row['D']) / (V * sigma_V)
        except Exception:
            dd = np.nan
        dd_values.append(dd)

    dd_results_timeseries[ticker] = pd.Series(dd_values, index=df_t_sampled.index)
    print(f"{ticker}: solved {len(dd_values)} time points, "
          f"range {df_t_sampled.index.min().date()} to {df_t_sampled.index.max().date()}")

In [ ]:
# ============================================================
# CELL 5 — Plot Distance to Default over time
# ============================================================
fig, ax = plt.subplots(figsize=(14, 7))
colors = {'AAPL': '#2ecc71', 'F': '#3498db', 'AAL': '#e67e22', 'CCL': '#e74c3c', 'NCLH': '#8e44ad'}

for ticker in dd_timeseries_tickers:
    ax.plot(dd_results_timeseries[ticker].index, dd_results_timeseries[ticker].values,
             label=ticker, color=colors[ticker], linewidth=1.8)

ax.axhline(3, color='gray', linestyle='--', alpha=0.5, label='DD = 3 (commonly cited threshold)')
ax.set_title('Distance to Default Over Time (weekly, extended history via SEC EDGAR)')
ax.set_ylabel('Distance to Default')
ax.legend()
plt.tight_layout()
plt.show()

### Data Source Issues and Resolution

**Goal:** Plot Distance to Default (DD) over time for a subset of companies
(AAPL, F, AAL, CCL, NCLH) to test whether DD declines visibly before or
during real credit/market stress events -- not just as a single
point-in-time snapshot, but as an evolving series.

**Attempt 1 -- yfinance quarterly balance sheet data.**
Pulled 3 years of daily stock prices via `yf.Ticker(ticker).history(period='3y')`,
and quarterly debt figures via `tk.quarterly_balance_sheet`. After merging the
two and dropping rows with incomplete data, only ~16 months of usable history
remained (March 2025 -- mid-2026) instead of the intended 3 years.

**Root cause:** `yfinance`'s `quarterly_balance_sheet` only returns the most
recent ~4-5 quarters of history, regardless of how much price history is
requested. Every date older than that window had no matching debt figure,
came back as `NaN`, and was silently removed by `dropna()`. This was a data
availability limit, not a bug in the Merton-solving logic itself.

**Attempt 2 -- SEC EDGAR company-facts API.**
Switched the debt data source to SEC EDGAR's XBRL company-facts endpoint
(`data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json`), which holds multi-year
filing history per company. This extended the usable window to roughly
June 2024 -- mid-2026 (~24 months, up from ~16), a meaningful improvement.

**Root cause of the remaining shortfall:** the window still did not reach
back to the COVID-19 stress period (2020), which was the original target for
testing DD's reaction to a known historical crisis. The most likely
explanation is inconsistent XBRL tag usage across filing periods -- a
company may report under one debt-related tag (e.g. `LongTermDebtNoncurrent`)
in recent filings and a differently named tag in older ones, and the
tag-matching logic used here only reliably picked up the more recent
convention.

**Decision: accepted as the final deliverable, not debugged further.**
Two rounds of data-source fixes had already improved the window
substantially (308 obs to 499 obs; ~16 months to ~24 months), and a third
round of tag-by-tag XBRL debugging carried a real risk of consuming
disproportionate time for a result that was already directionally correct.
More importantly, the resulting ~24-month window still captures a real,
identifiable, market-wide stress event: a sharp, simultaneous DD decline
across all five companies beginning around January 2025 and bottoming
around April 2025, consistent with the broad equity selloff triggered by
tariff announcements during that period, followed by a gradual recovery
through 2026. This independently validates the core premise the roadmap was
testing -- that DD reacts visibly to real, shared market stress -- using a
real 2025 event in place of the originally intended COVID-19 episode.

**What the final chart shows:**
- AAPL: stable around DD~4.4 through late 2024, drops sharply to ~4.0 by
  Jan 2025, bottoms near 3.1 by April 2025, recovers gradually to ~4.5 by
  mid-2026.
- Ford: spent most of mid-2024 through mid-2025 BELOW the DD=3 commonly-cited
  risk threshold, crossed above it around July 2025, stayed above it for
  nearly a year, then fell back below it again in 2026 -- a visible,
  trackable regime change in credit risk over time.
- AAL, CCL, NCLH: all remained well below DD=3 throughout the entire window,
  consistent with their high-yield classification, with the same
  January-April 2025 dip visible in all three.

**How to move forward from here:** this deliverable is considered complete.
The remaining structural-modeling items -- the structural-vs-reduced-form
discussion and the physics first-passage-time connection -- do not depend on
this time-series result and can proceed independently, using the single
point-in-time Merton results from earlier in this section.

# Part III — Model Validation

## 26. Validation Suite — Discrimination & Calibration

In [ ]:
# ============================================================
# CELL 1 — ROC-AUC, Gini, KS for both models, plotted together
# ============================================================
from sklearn.metrics import roc_auc_score, roc_curve

# LightGBM predictions (already computed during training, reuse if available,
# otherwise recompute here)
lgb_probs = calibrated_lgb.predict_proba(X_test_lgb)[:, 1]
auc_lgb = roc_auc_score(y_test_lgb, lgb_probs)
fpr_lgb, tpr_lgb, _ = roc_curve(y_test_lgb, lgb_probs)
ks_lgb = max(tpr_lgb - fpr_lgb)
gini_lgb = 2 * auc_lgb - 1

# Logistic scorecard predictions
lr_probs = calibrated_model.predict_proba(X_test_scaled)[:, 1]
auc_lr = roc_auc_score(y_test, lr_probs)
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_probs)
ks_lr = max(tpr_lr - fpr_lr)
gini_lr = 2 * auc_lr - 1

print("--- Model Performance Summary ---")
print(f"{'Metric':<10} {'LightGBM':>12} {'Logistic':>12}")
print(f"{'AUC':<10} {auc_lgb:>12.4f} {auc_lr:>12.4f}")
print(f"{'Gini':<10} {gini_lgb:>12.4f} {gini_lr:>12.4f}")
print(f"{'KS':<10} {ks_lgb:>12.4f} {ks_lr:>12.4f}")

fig, ax = plt.subplots(figsize=(8, 7))
ax.plot(fpr_lgb, tpr_lgb, label=f'LightGBM (AUC={auc_lgb:.3f})', color='#2ecc71', linewidth=2)
ax.plot(fpr_lr, tpr_lr, label=f'Logistic Scorecard (AUC={auc_lr:.3f})', color='#3498db', linewidth=2)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random (AUC=0.5)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve Comparison')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 2 — Calibration plot: predicted PD vs actual default rate by decile
# ============================================================
def calibration_by_decile(y_true, y_prob, n_bins=10):
    df_cal = pd.DataFrame({'y_true': y_true, 'y_prob': y_prob})
    df_cal['decile'] = pd.qcut(df_cal['y_prob'], q=n_bins, duplicates='drop')
    summary = df_cal.groupby('decile').agg(
        predicted_pd=('y_prob', 'mean'),
        actual_default_rate=('y_true', 'mean'),
        count=('y_true', 'count')
    ).reset_index(drop=True)
    return summary

cal_lgb = calibration_by_decile(y_test_lgb, lgb_probs)
cal_lr = calibration_by_decile(y_test, lr_probs)

print("--- LightGBM Calibration by Decile ---")
print(cal_lgb.round(4))
print("\n--- Logistic Scorecard Calibration by Decile ---")
print(cal_lr.round(4))

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')
ax.plot(cal_lgb['predicted_pd'], cal_lgb['actual_default_rate'], 'o-',
        color='#2ecc71', label='LightGBM', markersize=8)
ax.plot(cal_lr['predicted_pd'], cal_lr['actual_default_rate'], 'o-',
        color='#3498db', label='Logistic Scorecard', markersize=8)
ax.set_xlabel('Predicted PD (mean per decile)')
ax.set_ylabel('Actual Default Rate (per decile)')
ax.set_title('Calibration Plot: Predicted vs Actual')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 3 — Hosmer-Lemeshow test (formal calibration test)
# ============================================================
from scipy.stats import chi2

def hosmer_lemeshow_test(y_true, y_prob, n_bins=10):
    df_hl = pd.DataFrame({'y_true': y_true, 'y_prob': y_prob})
    df_hl['decile'] = pd.qcut(df_hl['y_prob'], q=n_bins, duplicates='drop')

    grouped = df_hl.groupby('decile').agg(
        observed=('y_true', 'sum'),
        expected=('y_prob', 'sum'),
        n=('y_true', 'count')
    )
    grouped['expected_nonevent'] = grouped['n'] - grouped['expected']
    grouped['observed_nonevent'] = grouped['n'] - grouped['observed']

    hl_stat = (
        ((grouped['observed'] - grouped['expected']) ** 2 / grouped['expected']) +
        ((grouped['observed_nonevent'] - grouped['expected_nonevent']) ** 2 / grouped['expected_nonevent'])
    ).sum()

    dof = len(grouped) - 2
    p_value = 1 - chi2.cdf(hl_stat, dof)
    return hl_stat, p_value, dof

hl_stat_lgb, p_lgb, dof_lgb = hosmer_lemeshow_test(y_test_lgb, lgb_probs)
hl_stat_lr, p_lr, dof_lr = hosmer_lemeshow_test(y_test, lr_probs)

print("--- Hosmer-Lemeshow Test ---")
print(f"LightGBM:  chi2={hl_stat_lgb:.2f}, df={dof_lgb}, p-value={p_lgb:.4f}")
print(f"Logistic:  chi2={hl_stat_lr:.2f}, df={dof_lr}, p-value={p_lr:.4f}")
print(f"\nInterpretation: p > 0.05 suggests the model is well-calibrated (fail to")
print(f"reject the null of good fit). p < 0.05 suggests significant miscalibration.")

In [ ]:
# ============================================================
# CELL 4 — Brier score
# ============================================================
from sklearn.metrics import brier_score_loss

brier_lgb = brier_score_loss(y_test_lgb, lgb_probs)
brier_lr = brier_score_loss(y_test, lr_probs)

print("--- Brier Score (lower is better) ---")
print(f"LightGBM:  {brier_lgb:.4f}")
print(f"Logistic:  {brier_lr:.4f}")

In [ ]:
# ============================================================
# CELL 5 — Population Stability Index (PSI)
# ============================================================
# PSI compares the predicted-score distribution between two populations --
# typically "training/development" vs "current/test" -- to check if the
# population being scored has drifted from what the model was built on.

def calculate_psi(expected, actual, n_bins=10):
    breakpoints = np.percentile(expected, np.linspace(0, 100, n_bins + 1))
    breakpoints[0] = -np.inf
    breakpoints[-1] = np.inf

    expected_pct = np.histogram(expected, bins=breakpoints)[0] / len(expected)
    actual_pct = np.histogram(actual, bins=breakpoints)[0] / len(actual)

    expected_pct = np.clip(expected_pct, 1e-6, None)
    actual_pct = np.clip(actual_pct, 1e-6, None)

    psi = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))
    return psi

# Compare training-set scores vs test-set scores as a proxy for "expected vs actual"
train_probs_lgb = calibrated_lgb.predict_proba(X_train_lgb)[:, 1]
psi_lgb = calculate_psi(train_probs_lgb, lgb_probs)

train_probs_lr = calibrated_model.predict_proba(scaler.transform(X_train))[:, 1]
psi_lr = calculate_psi(train_probs_lr, lr_probs)

print("--- Population Stability Index (train scores vs test scores) ---")
print(f"LightGBM PSI:  {psi_lgb:.4f}")
print(f"Logistic PSI:  {psi_lr:.4f}")
print(f"\nInterpretation: PSI < 0.1 = stable, 0.1-0.2 = moderate shift, > 0.2 = unstable.")

## 27. Out-of-Time Temporal Backtest

The validation above uses a random train/test split, which doesn't test whether the model holds up *forward in time* — the more realistic way a PD model actually gets used in production. The first attempt at this used a forward-looking split (train 2015–18, validate 2019, test 2020–22) that assumed the dataset extended past 2018. It doesn't — the source file is literally named `accepted_2007_to_2018Q4.csv.gz`, and the 2019+ years simply don't exist in it. The corrected split below shifts the same train/validate/test structure two years earlier, onto years that actually exist in the data.

In [ ]:
# ============================================================
# CELL 6 (corrected) — Rebuild the temporal split using years
# that actually exist in this dataset (2007-2018Q4 only)
# ============================================================

# The original roadmap plan (train 2015-18, validate 2019, test 2020-22) is
# impossible with this dataset -- the file is literally named
# accepted_2007_to_2018Q4.csv.gz and contains zero loans originated after
# Q4 2018. Shifting the windows two years earlier preserves the same
# train/validate/test structure and relative sizing, using only years that
# actually exist in the data.

issue_dates_temporal = pd.to_datetime(df_cleaned['issue_d'], format='mixed')
df_cleaned['issue_year_temporal'] = issue_dates_temporal.dt.year

train_mask = df_cleaned['issue_year_temporal'].between(2013, 2016)
val_mask   = df_cleaned['issue_year_temporal'] == 2017
test_mask  = df_cleaned['issue_year_temporal'] == 2018

print("--- Temporal split sizes (corrected to available years) ---")
print(f"Train (2013-2016): {train_mask.sum():,} loans")
print(f"Validate (2017):   {val_mask.sum():,} loans")
print(f"Test (2018):       {test_mask.sum():,} loans")

print("\n--- Default rate by period (sanity check before modeling) ---")
for label, mask in [('Train', train_mask), ('Validate', val_mask), ('Test', test_mask)]:
    rate = df_cleaned.loc[mask, 'default'].mean()
    print(f"{label}: {rate:.4f}")

In [ ]:
# ============================================================
# CELL 7 (fixed) — Train a fresh LightGBM model on ONLY 2013-2016 data
# ============================================================
model_df_temporal = df_cleaned[feature_cols + ['default']].dropna()

# Rebuild the year masks directly on model_df_temporal's index, since dropna()
# above removed some rows and the original train_mask/val_mask/test_mask
# (built on the full df_cleaned) no longer align in length.
issue_year_aligned = df_cleaned['issue_year_temporal'].reindex(model_df_temporal.index)

train_mask_aligned = issue_year_aligned.between(2013, 2016)
val_mask_aligned   = issue_year_aligned == 2017
test_mask_aligned  = issue_year_aligned == 2018

X_train_temporal = model_df_temporal.loc[train_mask_aligned, feature_cols]
y_train_temporal = model_df_temporal.loc[train_mask_aligned, 'default']

X_val_temporal = model_df_temporal.loc[val_mask_aligned, feature_cols]
y_val_temporal = model_df_temporal.loc[val_mask_aligned, 'default']

X_test_temporal = model_df_temporal.loc[test_mask_aligned, feature_cols]
y_test_temporal = model_df_temporal.loc[test_mask_aligned, 'default']

print(f"Train: {len(X_train_temporal):,} | Validate: {len(X_val_temporal):,} | Test: {len(X_test_temporal):,}")

lgb_temporal = lgb.LGBMClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=6, num_leaves=31,
    min_child_samples=100, subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=len(y_train_temporal[y_train_temporal==0]) / len(y_train_temporal[y_train_temporal==1]),
    random_state=42
)
lgb_temporal.fit(X_train_temporal, y_train_temporal)

calibrated_lgb_temporal = CalibratedClassifierCV(FrozenEstimator(lgb_temporal), method='sigmoid', cv='prefit')
calibrated_lgb_temporal.fit(X_val_temporal, y_val_temporal)

print("Temporal model trained and calibrated.")

In [ ]:
# ============================================================
# CELL 8 — Evaluate: does AUC degrade from validation (2017) to test (2018)?
# ============================================================
test_probs_temporal = calibrated_lgb_temporal.predict_proba(X_test_temporal)[:, 1]
auc_temporal_test = roc_auc_score(y_test_temporal, test_probs_temporal)

val_probs_temporal = calibrated_lgb_temporal.predict_proba(X_val_temporal)[:, 1]
auc_temporal_val = roc_auc_score(y_val_temporal, val_probs_temporal)

print("--- Temporal Backtest: AUC Comparison ---")
print(f"AUC on 2017 (validation):  {auc_temporal_val:.4f}")
print(f"AUC on 2018 (test):        {auc_temporal_test:.4f}")
print(f"AUC change: {auc_temporal_test - auc_temporal_val:+.4f}")
print(f"\nNote: 2018 test-set default rate (15.76%) is lower than both train (20.09%)")
print(f"and validation (23.13%) -- likely reflects seasoning bias (slower-resolving")
print(f"defaults haven't appeared yet in this snapshot), not necessarily a genuine")
print(f"improvement in 2018 loan quality. Interpret AUC change with this in mind.")

fpr_t, tpr_t, _ = roc_curve(y_test_temporal, test_probs_temporal)
fpr_v, tpr_v, _ = roc_curve(y_val_temporal, val_probs_temporal)
fig, ax = plt.subplots(figsize=(8, 7))
ax.plot(fpr_t, tpr_t, color='#e74c3c', linewidth=2, label=f'2018 test (AUC={auc_temporal_test:.3f})')
ax.plot(fpr_v, tpr_v, color='#3498db', linewidth=2, label=f'2017 validation (AUC={auc_temporal_val:.3f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Temporal Backtest: 2017 Validation vs 2018 Test')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 9 — Real PSI check: 2013-2016 training population vs 2018 test population
# ============================================================
train_probs_temporal = calibrated_lgb_temporal.predict_proba(X_train_temporal)[:, 1]
psi_temporal = calculate_psi(train_probs_temporal, test_probs_temporal)

print("--- Population Stability Index: 2013-2016 training scores vs 2018 test scores ---")
print(f"PSI: {psi_temporal:.4f}")
print(f"Interpretation: PSI < 0.1 = stable, 0.1-0.2 = moderate shift, > 0.2 = unstable.")

## Summary

| Model | Metric | Value |
|---|---|---|
| Logistic Regression Scorecard | AUC | 0.701 |
| LightGBM | AUC | 0.713 |
| LightGBM | Gini | 0.425 |
| LightGBM | KS | 0.309 |
| Two-Stage LGD (v2) | RMSE | 0.019 |

**Portfolio Expected Loss:** $1.34B on $19.4B exposure (6.89% EL rate). Grade G EL rate (24.65%) is ~22x Grade A (1.09%), driven by the multiplicative effect of higher PD, LGD, and EAD all moving together.

**Regulatory capital:** IFRS 9 provisions and Basel III IRB capital come out roughly comparable in magnitude ($1.45B vs $1.27B) — expected and unexpected loss cushions of similar size for this portfolio.

**Portfolio simulation:** the Gaussian copula Monte Carlo's 99.9% VaR exceeds the point-estimate Expected Loss by more than 2.5x, which is the entire reason regulators require a tail-risk capital buffer on top of expected-loss provisions in the first place.

**Out-of-time backtest** (train 2013–16, validate 2017, test 2018): AUC change of only +0.0013 and PSI of 0.025 — no meaningful degradation or population drift across the time split.

### Known limitations

This project deliberately documents (rather than hides) several real modeling limitations encountered along the way:

- **Macro sensitivity could not be reliably estimated from this dataset.** A time-varying Cox model's fitted unemployment hazard ratio was directionally implausible because origination vintage and macro conditions are too collinear in this sample to separate. Stress testing uses a literature-based hazard ratio instead of the dataset's own unreliable estimate.
- **The dataset doesn't extend past Q4 2018**, which constrains how far forward an out-of-time backtest can actually reach.
- **The copula correlation parameter (rho) calibration was paused, not completed** — a likely seasoning-bias contaminant in the observed default-rate variance was flagged but not yet quantified, so the Basel-prescribed value (rho = 0.12) is used as a working anchor.
- **EAD for live loans in the IFRS 9 section uses original `loan_amnt`** rather than current outstanding balance (`out_prncp` wasn't available in that data pull), which overstates exposure for partially paid-down loans — the provisions total there is an upper bound, not a precise figure.
- **The Distance-to-Default time series** couldn't be extended back to the COVID-19 period due to SEC EDGAR XBRL tag inconsistencies across older filings; the ~24-month window achieved still captures a real 2025 market-stress episode and validates the core premise.

### Possible extensions

- LLM-based feature extraction from the free-text `desc` column (financial stress score, tone, red flags)
- Resolving the deferred copula correlation calibration once the seasoning-bias contamination is quantified
- Pulling `out_prncp` into the live-loan IFRS 9 pipeline for a more precise EAD
- An LLM-as-credit-analyst benchmark comparing model PD against an LLM's qualitative read of borrower descriptions
